# JungleSurvivor — Kaggle 整合測試 Notebook

**叢林求生離線 AI 助手 — 四層辨識 Pipeline 完整測試**

本 Notebook 整合了 JungleSurvivor 的所有模組，在 Kaggle GPU 環境下測試完整的辨識流程。

**環境需求：**
- Kaggle Notebook + GPU T4 x2
- Gemma 4 E2B 模型存取權限

**測試項目：**
1. 危險植物快篩（姑婆芋）
2. 多照片辨識提升信心度
3. 蛇類辨識
4. 可食用植物辨識（山蘇）

In [ ]:
# Cell 1: 安裝最新版 Transformers（Gemma 4 需要）
!pip install -q --upgrade transformers accelerate

# 匯入相依套件
import json
import re
import os
import torch
import requests
from io import BytesIO
from PIL import Image
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime
from transformers import AutoProcessor, AutoModelForImageTextToText

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"         VRAM: {mem:.1f} GB")

In [ ]:
# Cell 2: 全域設定 (config)

MODEL_ID = "google/gemma-4-E2B-it"
MAX_NEW_TOKENS = 4096
DEFAULT_DTYPE = "bfloat16"

THRESHOLDS = {
    "danger_screening": 60,
    "confusion_pairs": 80,
    "useful_resources": 70,
}

DEFAULT_REGION = "east_asia_subtropical"

JSON_START_MARKER = "<JSON_START>"
JSON_END_MARKER = "<JSON_END>"

WARNING_LEVELS = {
    "RED":    {"color": "#FF0000", "label_zh": "🔴 危險", "action": "立即遠離"},
    "YELLOW": {"color": "#FFD700", "label_zh": "🟡 注意", "action": "謹慎處理"},
    "GREEN":  {"color": "#00AA00", "label_zh": "🟢 安全", "action": "可安全使用"},
    "GRAY":   {"color": "#888888", "label_zh": "⚪ 不確定", "action": "無法判斷，建議不碰"},
}

print("✅ Config 載入完成")

## 知識庫（嵌入式）

將所有 JSON 知識庫嵌入 Notebook，不需要外部檔案。

In [ ]:
import json

# Embedded knowledge base JSON (east_asia_subtropical + emergency)

_TOXIC_PLANTS_JSON = r'''
[
  {
    "id": "alocasia_odora",
    "scientific_name": "Alocasia odora",
    "common_names": { "zh-TW": "姑婆芋", "en": "Giant Elephant Ear" },
    "danger_level": "high",
    "toxicity": "全株有毒，汁液含草酸鈣針晶，誤食致口腔灼傷、腹瀉、喉嚨腫脹",
    "morphology": {
      "leaf_shape": "大型心形或箭形，長 50-90cm，寬 40-60cm",
      "leaf_surface": "光滑發亮，深綠色，水珠不成珠狀（會攤平）",
      "leaf_venation": "明顯的羽狀脈，主脈粗壯突出",
      "leaf_tip": "朝上或水平指向",
      "petiole": "粗壯，長 50-100cm，綠色常帶紫暈，接在葉片邊緣",
      "stem": "粗短直立莖，多汁",
      "flower": "佛焰苞花序，黃綠色",
      "fruit": "紅色漿果串"
    },
    "habitat": "林下、溪邊、潮濕陰暗處，海拔 0-1500m",
    "confusion_with": ["colocasia_esculenta"],
    "affected_parts": ["全株", "汁液"],
    "symptoms": ["口腔灼傷", "舌頭麻痺", "腹瀉", "喉嚨腫脹", "嘔吐"],
    "first_aid": "立即用大量清水漱口，勿催吐，儘速就醫。皮膚接觸汁液時以大量清水沖洗。",
    "distribution": {
      "altitude_range": [0, 1500],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  },
  {
    "id": "cerbera_manghas",
    "scientific_name": "Cerbera manghas",
    "common_names": { "zh-TW": "海檬果", "en": "Sea Mango" },
    "danger_level": "critical",
    "toxicity": "全株有毒，種子劇毒，含海檬果毒素（cerberin），誤食可致心臟麻痺死亡",
    "morphology": {
      "leaf_shape": "倒披針形或長橢圓形，長 15-30cm，寬 3-5cm，革質",
      "leaf_surface": "光滑，深綠色，有光澤",
      "leaf_venation": "側脈明顯，平行排列",
      "leaf_tip": "漸尖",
      "petiole": "短，長 1-3cm",
      "stem": "直立喬木，高可達 8-15m，斷面有白色乳汁",
      "flower": "白色五瓣花，中心粉紅色，有香味，聚繖花序",
      "fruit": "卵圓形核果，直徑 5-7cm，青綠轉暗紅，內含劇毒種子"
    },
    "habitat": "海岸、河口、紅樹林邊緣、公園（常作景觀樹）",
    "confusion_with": [],
    "affected_parts": ["全株", "種子（劇毒）", "乳汁"],
    "symptoms": ["嘔吐", "腹痛", "心律不整", "心臟麻痺", "可致死"],
    "first_aid": "立即催吐（若意識清醒），儘速送醫，告知醫師疑似海檬果中毒。此為心臟毒性，需緊急處理。",
    "distribution": {
      "altitude_range": [0, 100],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  },
  {
    "id": "datura_stramonium",
    "scientific_name": "Datura stramonium",
    "common_names": { "zh-TW": "曼陀羅", "en": "Jimsonweed" },
    "danger_level": "critical",
    "toxicity": "全株劇毒，含莨菪鹼（scopolamine）、東莨菪鹼（atropine），誤食致譫妄、幻覺、死亡",
    "morphology": {
      "leaf_shape": "卵形，長 8-20cm，邊緣有粗鋸齒或不規則裂片",
      "leaf_surface": "略粗糙，暗綠色，搓揉後有臭味",
      "leaf_venation": "網狀脈",
      "leaf_tip": "銳尖",
      "petiole": "長 3-6cm",
      "stem": "直立草本或亞灌木，高 50-150cm，莖綠色或紫色，光滑",
      "flower": "白色或淡紫色喇叭形大花，長 6-10cm，單生於枝腋",
      "fruit": "球形蒴果，直徑 3-5cm，外被硬刺（關鍵特徵）"
    },
    "habitat": "荒地、路旁、廢棄地、農田邊緣",
    "confusion_with": [],
    "affected_parts": ["全株", "種子（濃度最高）", "葉"],
    "symptoms": ["瞳孔放大", "口乾", "心跳加速", "幻覺", "譫妄", "抽搐", "昏迷", "可致死"],
    "first_aid": "立即催吐（若意識清醒且在 1 小時內），儘速送醫。注意維持呼吸道暢通。",
    "distribution": {
      "altitude_range": [0, 1000],
      "climate_zones": ["亞熱帶", "溫帶", "熱帶"]
    }
  },
  {
    "id": "urtica_thunbergiana",
    "scientific_name": "Urtica thunbergiana",
    "common_names": { "zh-TW": "咬人貓", "en": "Stinging Nettle (Taiwan)" },
    "danger_level": "medium",
    "toxicity": "莖葉有透明焮刺毛，觸碰注入蟻酸等化學物質，引起劇烈灼痛，持續數小時",
    "morphology": {
      "leaf_shape": "闊卵形至心形，長 6-13cm，邊緣有粗鋸齒",
      "leaf_surface": "兩面均有焮刺毛（透明針狀突起，肉眼可見）——關鍵特徵",
      "leaf_venation": "網狀脈，3-5 條基出脈",
      "leaf_tip": "銳尖",
      "petiole": "長 3-8cm，有刺毛",
      "stem": "直立草本，高 50-120cm，莖上也有刺毛，方形莖",
      "flower": "穗狀花序，小型綠色花",
      "fruit": "瘦果，微小"
    },
    "habitat": "中海拔山區林下，潮濕陰暗處，海拔 500-3000m",
    "confusion_with": [],
    "affected_parts": ["莖", "葉", "焮刺毛"],
    "symptoms": ["劇烈灼痛", "皮膚紅腫", "丘疹", "疼痛持續數小時至一天"],
    "first_aid": "用膠帶黏除皮膚上的刺毛。冷水沖洗患處。可塗抹含類固醇的止癢藥膏。避免搔抓。",
    "distribution": {
      "altitude_range": [500, 3000],
      "climate_zones": ["亞熱帶", "溫帶"]
    }
  },
  {
    "id": "dendrocnide_meyeniana",
    "scientific_name": "Dendrocnide meyeniana",
    "common_names": { "zh-TW": "咬人狗", "en": "Stinging Tree (Taiwan)" },
    "danger_level": "high",
    "toxicity": "焮刺毛含蟻酸等毒素，觸碰引起劇烈灼痛，疼痛可持續數週至數月",
    "morphology": {
      "leaf_shape": "大型卵形至圓形，長 15-30cm，寬 10-20cm，全緣或微波狀",
      "leaf_surface": "正面光滑或疏被短毛，背面密生刺毛（關鍵特徵）",
      "leaf_venation": "明顯的三出脈",
      "leaf_tip": "鈍尖或銳尖",
      "petiole": "長 5-15cm",
      "stem": "灌木至小喬木，高 2-5m，莖上有刺毛",
      "flower": "聚繖花序，小型淡綠色花",
      "fruit": "小型核果"
    },
    "habitat": "低海拔闊葉林，林緣或開闊地，海拔 0-800m",
    "confusion_with": [],
    "affected_parts": ["全株表面（刺毛）"],
    "symptoms": ["劇烈灼痛", "持續疼痛數週至數月", "皮膚紅腫", "嚴重者可能過敏性休克"],
    "first_aid": "用膠帶反覆黏除刺毛。冷水沖洗。勿用手觸摸患處。疼痛持續超過一天應就醫。",
    "distribution": {
      "altitude_range": [0, 800],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  },
  {
    "id": "chlorophyllum_molybdites",
    "scientific_name": "Chlorophyllum molybdites",
    "common_names": { "zh-TW": "綠褶菇", "en": "Green-spored Parasol" },
    "danger_level": "high",
    "toxicity": "含多種毒素，誤食致嚴重腸胃炎，為台灣最常見的誤食毒菇",
    "morphology": {
      "cap": "傘蓋直徑 10-30cm，初球形後展平，白色至淡褐色，中央有褐色鱗片",
      "gills": "初白色，成熟後轉為綠色（關鍵特徵）",
      "stipe": "菌柄長 10-25cm，白色，基部膨大，有可移動的菌環",
      "spore_print": "綠色（關鍵特徵）",
      "flesh": "白色，受傷後不變色或微變褐色"
    },
    "habitat": "草地、公園、草坪上群生，雨後大量出現，海拔 0-500m",
    "confusion_with": ["macrolepiota_procera"],
    "affected_parts": ["子實體（全株）"],
    "symptoms": ["嚴重嘔吐", "腹瀉", "腹痛", "脫水", "症狀在食後 1-3 小時出現"],
    "first_aid": "催吐（若在食後 1 小時內），大量補充水分防止脫水，儘速就醫。保留殘餘菇體供醫師辨識。",
    "distribution": {
      "altitude_range": [0, 500],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  },
  {
    "id": "amanita_phalloides",
    "scientific_name": "Amanita phalloides",
    "common_names": { "zh-TW": "鱗柄白毒鵝膏（死帽蕈近似種）", "en": "Death Cap (related species)" },
    "danger_level": "critical",
    "toxicity": "含鵝膏毒素（amatoxin），致死量極低，誤食致肝腎衰竭，死亡率極高",
    "morphology": {
      "cap": "傘蓋直徑 5-15cm，白色至淡綠色或淡黃色，光滑，幼時半球形後展平",
      "gills": "白色，離生，密集排列",
      "stipe": "菌柄長 8-15cm，白色，基部有明顯的菌托（杯狀，關鍵特徵）",
      "ring": "菌柄上部有白色薄膜狀菌環",
      "spore_print": "白色",
      "flesh": "白色，無特殊氣味（危險！易被認為無毒）"
    },
    "habitat": "闊葉林或針闊混合林下，常在樹根附近，海拔 500-2500m",
    "confusion_with": [],
    "affected_parts": ["子實體（全株）"],
    "symptoms": ["潛伏期 6-24 小時（危險！初期無症狀）", "嘔吐腹瀉（假性恢復期後加重）", "肝腎衰竭", "黃疸", "多器官衰竭", "死亡"],
    "first_aid": "立即送醫！告知疑似鵝膏毒素中毒。保留殘餘菇體。此毒素無特效解毒劑，需緊急支持性治療。時間是關鍵。",
    "distribution": {
      "altitude_range": [500, 2500],
      "climate_zones": ["亞熱帶", "溫帶"]
    }
  },
  {
    "id": "toxicodendron_vernicifluum",
    "scientific_name": "Toxicodendron vernicifluum",
    "common_names": { "zh-TW": "漆樹", "en": "Chinese Lacquer Tree" },
    "danger_level": "medium",
    "toxicity": "樹液含漆酚（urushiol），接觸致嚴重過敏性接觸性皮膚炎",
    "morphology": {
      "leaf_shape": "奇數羽狀複葉，小葉 7-13 枚，每枚長 7-15cm，卵形至橢圓形",
      "leaf_surface": "幼葉微被毛，成熟葉近光滑，秋季轉紅（漂亮但危險）",
      "leaf_venation": "羽狀脈",
      "leaf_tip": "漸尖",
      "petiole": "葉軸長 20-40cm",
      "stem": "落葉喬木，高 10-20m，樹皮灰白色，切割後流出白色漆液（接觸即過敏）",
      "flower": "圓錐花序，小型黃綠色花",
      "fruit": "扁球形核果，淡黃色"
    },
    "habitat": "中低海拔闊葉林，山坡地，海拔 200-2500m",
    "confusion_with": [],
    "affected_parts": ["樹液（漆液）", "枝葉"],
    "symptoms": ["嚴重皮膚紅腫", "水泡", "劇烈搔癢", "過敏者可能全身性反應"],
    "first_aid": "立即用大量肥皂水清洗接觸部位（15分鐘內最有效）。勿搔抓水泡。塗抹含類固醇藥膏。嚴重者就醫。",
    "distribution": {
      "altitude_range": [200, 2500],
      "climate_zones": ["亞熱帶", "溫帶"]
    }
  },
  {
    "id": "nerium_oleander",
    "scientific_name": "Nerium oleander",
    "common_names": { "zh-TW": "夾竹桃", "en": "Oleander" },
    "danger_level": "critical",
    "toxicity": "全株劇毒，含強心苷（oleandrin），誤食極少量即可致心臟停跳死亡",
    "morphology": {
      "leaf_shape": "線狀披針形，長 10-20cm，寬 1-2.5cm，革質，葉緣全緣略反捲",
      "leaf_surface": "深綠色，光滑，葉背有明顯中脈",
      "leaf_venation": "羽狀側脈密集，近平行",
      "leaf_arrangement": "三葉輪生（關鍵特徵）",
      "petiole": "極短",
      "stem": "常綠灌木，高 2-5m，分枝多，枝條灰綠色",
      "flower": "粉紅、白或紅色，五瓣，漏斗形，聚繖花序，有芳香",
      "fruit": "細長蓇葖果，長 10-20cm，成熟開裂露出帶毛種子"
    },
    "habitat": "公園、行道樹、庭園（常見觀賞植物），河岸，海拔 0-500m",
    "confusion_with": [],
    "affected_parts": ["全株（包括乾枯的葉和枝）", "花蜜", "燃燒煙霧也有毒"],
    "symptoms": ["噁心嘔吐", "腹痛", "心律不整", "心跳過慢", "視覺異常", "心臟停跳", "可致死"],
    "first_aid": "立即催吐（若意識清醒），儘速送醫。此為心臟毒性，需持續心電圖監測。勿用夾竹桃枝條烤食物。",
    "distribution": {
      "altitude_range": [0, 500],
      "climate_zones": ["亞熱帶", "熱帶", "溫帶"]
    }
  },
  {
    "id": "cycas_revoluta",
    "scientific_name": "Cycas revoluta",
    "common_names": { "zh-TW": "蘇鐵（鐵樹）", "en": "Sago Palm" },
    "danger_level": "high",
    "toxicity": "種子和嫩葉含蘇鐵苷（cycasin），誤食致肝臟損害，有致癌性",
    "morphology": {
      "leaf_shape": "大型羽狀複葉，長 50-150cm，小葉線形，硬挺，邊緣反捲",
      "leaf_surface": "深綠色，革質，有光澤",
      "leaf_venation": "小葉僅一條中脈",
      "leaf_arrangement": "叢生於莖頂，向外輻射展開（棕櫚狀）",
      "petiole": "葉柄基部有刺",
      "stem": "粗壯圓柱形樹幹，表面有葉痕，高 1-3m（生長極慢）",
      "flower": "雌雄異株，雄花序圓柱形直立，雌花序扁球形",
      "fruit": "種子球形，直徑 3-4cm，成熟時橙紅色"
    },
    "habitat": "公園、庭園（觀賞植物），山坡地，海拔 0-500m",
    "confusion_with": [],
    "affected_parts": ["種子（毒性最高）", "嫩葉", "根"],
    "symptoms": ["嘔吐", "腹瀉", "肝臟損害（延遲性）", "黃疸", "長期接觸有致癌風險"],
    "first_aid": "催吐（若在食後 1 小時內），儘速送醫。需追蹤肝功能指標。",
    "distribution": {
      "altitude_range": [0, 500],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  }
]
'''
TOXIC_PLANTS = json.loads(_TOXIC_PLANTS_JSON)

_CONFUSION_PAIRS_JSON = r'''
[
  {
    "id": "taro_vs_alocasia",
    "safe_species": {
      "id": "colocasia_esculenta",
      "scientific_name": "Colocasia esculenta",
      "common_name_zh": "芋頭",
      "common_name_en": "Taro",
      "key_features": {
        "leaf_surface": "有天鵝絨般的微絨毛質感，不太光滑",
        "water_droplet_test": "水珠在葉面呈圓珠狀滾動（荷葉效應）",
        "leaf_tip": "通常朝下垂",
        "petiole_attachment": "接在葉片邊緣內側（盾狀著生）",
        "petiole_color": "綠色，較少紫暈",
        "leaf_size": "較小，葉長 20-50cm",
        "underground": "有圓形塊莖，可食用"
      }
    },
    "dangerous_species": {
      "id": "alocasia_odora",
      "scientific_name": "Alocasia odora",
      "common_name_zh": "姑婆芋",
      "common_name_en": "Giant Elephant Ear",
      "danger_level": "high",
      "key_features": {
        "leaf_surface": "光滑發亮，質地像打了蠟",
        "water_droplet_test": "水珠攤平不成珠狀（不具荷葉效應）",
        "leaf_tip": "朝上或水平指向",
        "petiole_attachment": "接在葉片邊緣（非盾狀）",
        "petiole_color": "綠色常帶紫暈",
        "leaf_size": "較大，葉長 50-90cm",
        "underground": "根莖粗短，有毒"
      }
    },
    "lethality": "high",
    "comparison_table": [
      { "feature": "葉面質感", "safe": "微絨毛，不光滑", "danger": "光滑發亮，像打蠟" },
      { "feature": "水珠測試", "safe": "水珠成圓珠狀滾動", "danger": "水珠攤平" },
      { "feature": "葉尖方向", "safe": "朝下垂", "danger": "朝上或水平" },
      { "feature": "葉柄接合", "safe": "葉片內側（盾狀）", "danger": "葉片邊緣" },
      { "feature": "葉片大小", "safe": "較小（20-50cm）", "danger": "較大（50-90cm）" },
      { "feature": "葉柄紫暈", "safe": "少", "danger": "常見" }
    ],
    "interactive_tests": [
      {
        "test_name": "水珠測試",
        "priority": 1,
        "is_critical": true,
        "instruction_zh": "在葉面滴一滴水，觀察水珠的形狀",
        "safe_result": "水珠呈圓珠狀，可以滾動 → 較可能是芋頭",
        "danger_result": "水珠攤平，不成珠狀 → 較可能是姑婆芋"
      },
      {
        "test_name": "葉柄接合處特寫",
        "priority": 2,
        "is_critical": true,
        "instruction_zh": "拍攝葉柄與葉片的接合處特寫",
        "safe_result": "葉柄接在葉片邊緣內側（盾狀著生）",
        "danger_result": "葉柄接在葉片邊緣"
      },
      {
        "test_name": "葉面質感特寫",
        "priority": 3,
        "is_critical": false,
        "instruction_zh": "拍攝葉面特寫，觀察是否光滑如蠟，還是有微絨毛？",
        "safe_result": "有微絨毛，不光滑",
        "danger_result": "光滑發亮，像打了蠟"
      }
    ]
  },
  {
    "id": "parasol_vs_green_spored",
    "safe_species": {
      "id": "macrolepiota_procera",
      "scientific_name": "Macrolepiota procera",
      "common_name_zh": "高大環柄菇",
      "common_name_en": "Parasol Mushroom",
      "key_features": {
        "cap": "傘蓋直徑 10-30cm，表面有褐色鱗片，中央較深",
        "gills": "白色，成熟後仍保持白色或淡奶油色",
        "stipe": "菌柄細長，有蛇皮狀斑紋（關鍵特徵）",
        "ring": "雙層菌環，可上下滑動",
        "spore_print": "白色"
      }
    },
    "dangerous_species": {
      "id": "chlorophyllum_molybdites",
      "scientific_name": "Chlorophyllum molybdites",
      "common_name_zh": "綠褶菇",
      "common_name_en": "Green-spored Parasol",
      "danger_level": "high",
      "key_features": {
        "cap": "傘蓋直徑 10-30cm，外觀與高大環柄菇極似",
        "gills": "初白色，成熟後轉為綠色（關鍵特徵）",
        "stipe": "菌柄較粗，缺乏蛇皮狀斑紋",
        "ring": "菌環較薄",
        "spore_print": "綠色（關鍵特徵）"
      }
    },
    "lethality": "medium",
    "comparison_table": [
      { "feature": "菌褶顏色", "safe": "始終白色或奶油色", "danger": "成熟後轉綠色" },
      { "feature": "孢子印顏色", "safe": "白色", "danger": "綠色" },
      { "feature": "菌柄紋路", "safe": "有蛇皮狀斑紋", "danger": "無蛇皮斑紋" },
      { "feature": "菌環", "safe": "雙層，可移動", "danger": "較薄，單層" }
    ],
    "interactive_tests": [
      {
        "test_name": "菌褶顏色檢查",
        "priority": 1,
        "is_critical": true,
        "instruction_zh": "小心翻開傘蓋，觀察菌褶（傘蓋下方的放射狀薄片）顏色",
        "safe_result": "菌褶為白色或淡奶油色",
        "danger_result": "菌褶帶有綠色或灰綠色"
      },
      {
        "test_name": "孢子印測試",
        "priority": 2,
        "is_critical": true,
        "instruction_zh": "將傘蓋放在白紙上靜置 2-4 小時，觀察掉落的孢子粉顏色",
        "safe_result": "白色孢子粉",
        "danger_result": "綠色或灰綠色孢子粉"
      }
    ]
  },
  {
    "id": "birds_nest_fern_vs_toxic_fern",
    "safe_species": {
      "id": "asplenium_nidus",
      "scientific_name": "Asplenium nidus",
      "common_name_zh": "山蘇（鳥巢蕨）",
      "common_name_en": "Bird's Nest Fern",
      "key_features": {
        "leaf_shape": "大型單葉，長披針形，長 50-120cm，寬 10-15cm",
        "leaf_arrangement": "叢生，從中心向外輻射展開，形成鳥巢狀（關鍵特徵）",
        "leaf_surface": "光滑，革質，全緣（無鋸齒）",
        "midrib": "明顯的黑褐色中肋",
        "sori": "葉背有線狀孢子囊群，沿側脈排列",
        "growth": "附生在樹幹上或岩石上"
      }
    },
    "dangerous_species": {
      "id": "toxic_fern_general",
      "scientific_name": "Various",
      "common_name_zh": "有毒蕨類（泛指）",
      "common_name_en": "Toxic Ferns",
      "danger_level": "low",
      "key_features": {
        "leaf_shape": "多為羽狀複葉（與山蘇的單葉明顯不同）",
        "leaf_arrangement": "通常不形成鳥巢狀",
        "leaf_surface": "可能有毛茸或不同質感",
        "growth": "多為地生型"
      }
    },
    "lethality": "low",
    "comparison_table": [
      { "feature": "葉形", "safe": "單葉，長披針形", "danger": "多為羽狀複葉" },
      { "feature": "排列方式", "safe": "鳥巢狀輻射展開", "danger": "無特定排列" },
      { "feature": "生長方式", "safe": "附生（樹幹/岩石）", "danger": "多地生" },
      { "feature": "中肋顏色", "safe": "黑褐色", "danger": "多為綠色" }
    ],
    "interactive_tests": [
      {
        "test_name": "葉背孢子囊觀察",
        "priority": 1,
        "is_critical": false,
        "instruction_zh": "翻開葉片背面，觀察是否有沿著側脈排列的線狀褐色條紋（孢子囊群）",
        "safe_result": "有整齊的線狀孢子囊群沿側脈排列",
        "danger_result": "孢子囊分佈不同或無孢子囊"
      }
    ]
  },
  {
    "id": "wild_ginger_vs_toxic_ginger",
    "safe_species": {
      "id": "hedychium_coronarium",
      "scientific_name": "Hedychium coronarium",
      "common_name_zh": "野薑花",
      "common_name_en": "White Ginger Lily",
      "key_features": {
        "flower": "白色蝴蝶形花朵，香氣濃郁（關鍵特徵）",
        "leaf_shape": "長橢圓形披針狀，長 20-40cm，互生排列",
        "rhizome": "根莖有薑味（關鍵特徵）",
        "stem": "直立草本，高 100-200cm"
      }
    },
    "dangerous_species": {
      "id": "toxic_zingiberaceae",
      "scientific_name": "Various Zingiberaceae",
      "common_name_zh": "有毒薑科植物",
      "common_name_en": "Toxic Ginger Family Plants",
      "danger_level": "medium",
      "key_features": {
        "flower": "花色可能不同（非純白蝴蝶形）",
        "rhizome": "根莖無薑味或有刺激性氣味",
        "leaf_shape": "葉形可能相似但細節不同"
      }
    },
    "lethality": "medium",
    "comparison_table": [
      { "feature": "花的形態", "safe": "白色蝴蝶形，強烈芳香", "danger": "花色或花形不同" },
      { "feature": "根莖氣味", "safe": "有明顯薑味", "danger": "無薑味或刺激性氣味" },
      { "feature": "花序位置", "safe": "頂生穗狀花序", "danger": "可能不同" }
    ],
    "interactive_tests": [
      {
        "test_name": "根莖氣味測試",
        "priority": 1,
        "is_critical": true,
        "instruction_zh": "小心刮開少量根莖表皮，聞是否有明顯的薑味",
        "safe_result": "有清新的薑味",
        "danger_result": "無薑味、有刺激性氣味、或有化學藥味"
      }
    ]
  },
  {
    "id": "termite_mushroom_vs_white_toxic",
    "safe_species": {
      "id": "termitomyces_sp",
      "scientific_name": "Termitomyces sp.",
      "common_name_zh": "雞肉絲菇",
      "common_name_en": "Termite Mushroom",
      "key_features": {
        "cap": "傘蓋灰白色至褐色，中央有尖突（關鍵特徵）",
        "gills": "白色至淡粉色",
        "stipe": "菌柄下方延伸入土中，連接白蟻巢（關鍵特徵）",
        "habitat": "只生長在白蟻巢上方",
        "texture": "撕開後呈絲狀（像雞肉絲，故名）"
      }
    },
    "dangerous_species": {
      "id": "white_toxic_mushroom",
      "scientific_name": "Amanita sp. / Leucoagaricus sp.",
      "common_name_zh": "白色毒菇類",
      "common_name_en": "White Toxic Mushrooms",
      "danger_level": "critical",
      "key_features": {
        "cap": "白色，可能無中央尖突",
        "gills": "白色",
        "stipe": "菌柄基部可能有菌托（杯狀）或膨大",
        "habitat": "非白蟻巢相關",
        "ring": "可能有薄膜狀菌環"
      }
    },
    "lethality": "critical",
    "comparison_table": [
      { "feature": "傘蓋中央", "safe": "有尖突", "danger": "無尖突或平滑" },
      { "feature": "菌柄基部", "safe": "深入土中連接白蟻巢", "danger": "有菌托（杯狀）" },
      { "feature": "生長基質", "safe": "白蟻巢上方（找蟻丘）", "danger": "林地、草地" },
      { "feature": "菌肉質地", "safe": "撕開呈絲狀", "danger": "撕開呈塊狀" }
    ],
    "interactive_tests": [
      {
        "test_name": "生長基質確認",
        "priority": 1,
        "is_critical": true,
        "instruction_zh": "觀察菇類生長的地面，是否有白蟻巢（土堆狀隆起）？小心挖開菌柄基部，是否深入土中？",
        "safe_result": "菌柄深入土中，下方有白蟻巢結構",
        "danger_result": "菌柄基部有杯狀菌托，或直接長在落葉/土壤上"
      },
      {
        "test_name": "菌肉撕裂測試",
        "priority": 2,
        "is_critical": false,
        "instruction_zh": "小心撕開一小塊菌肉，觀察質地",
        "safe_result": "呈絲狀撕開（像撕雞肉絲）",
        "danger_result": "呈塊狀或粉狀斷裂"
      }
    ]
  }
]
'''
CONFUSION_PAIRS = json.loads(_CONFUSION_PAIRS_JSON)

_EDIBLE_PLANTS_JSON = r'''
[
  {
    "id": "asplenium_nidus",
    "scientific_name": "Asplenium nidus",
    "common_names": { "zh-TW": "山蘇（鳥巢蕨）", "en": "Bird's Nest Fern" },
    "category": "edible",
    "edible_parts": ["嫩捲葉（中心尚未展開的幼葉）"],
    "morphology": {
      "leaf_shape": "大型單葉，長披針形，長 50-120cm，寬 10-15cm",
      "leaf_arrangement": "叢生，從中心向外輻射展開，形成鳥巢狀（關鍵特徵）",
      "leaf_surface": "光滑，革質，全緣（無鋸齒），鮮綠色",
      "midrib": "明顯的黑褐色中肋",
      "sori": "葉背有線狀孢子囊群，沿側脈排列",
      "growth_pattern": "附生在樹幹上或岩石上"
    },
    "habitat": "中低海拔潮濕森林，附生於大樹幹或岩壁，海拔 200-1500m",
    "harvesting": {
      "target": "只採中心尚未完全展開的嫩捲葉",
      "method": "用手或刀從基部折斷嫩芽，保留成熟葉確保植物持續生長",
      "season": "全年可採，春夏嫩芽最多",
      "sustainability": "每株最多採 1-2 支嫩芽，勿過度採集"
    },
    "preparation": {
      "method": "川燙後即可食用，也可快炒",
      "cooking_time": "川燙 1-2 分鐘",
      "taste": "口感滑嫩，微帶黏性",
      "notes": "可加薑絲、蒜末調味，是台灣原住民傳統野菜"
    },
    "nutrition": "富含維生素A、鈣、鐵、膳食纖維",
    "caution": "確認是山蘇而非其他蕨類。只食用嫩捲葉，成熟葉較硬不適合食用。",
    "confusion_risk": "low",
    "distribution": {
      "altitude_range": [200, 1500],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  },
  {
    "id": "diplazium_esculentum",
    "scientific_name": "Diplazium esculentum",
    "common_names": { "zh-TW": "過貓（過溝菜蕨）", "en": "Vegetable Fern" },
    "category": "edible",
    "edible_parts": ["嫩莖", "幼葉（捲曲的拳頭狀嫩芽）"],
    "morphology": {
      "leaf_shape": "二回羽狀複葉，成熟株高可達 1-2m",
      "leaf_arrangement": "叢生",
      "young_shoot": "頂端捲曲呈拳頭狀（蕨類典型特徵）——可食部位",
      "leaf_surface": "嫩葉綠色帶微紅，有細毛",
      "stem": "莖部黑褐色，有鱗片"
    },
    "habitat": "溪邊、潮濕地、水田旁，海拔 0-1000m",
    "harvesting": {
      "target": "嫩莖頂端捲曲處（約 15-20cm）",
      "method": "用手從柔軟處折斷",
      "season": "全年可採，春夏最嫩",
      "sustainability": "只採嫩芽，保留根部和成熟植株"
    },
    "preparation": {
      "method": "川燙或快炒，務必煮熟",
      "cooking_time": "川燙 2-3 分鐘",
      "taste": "口感滑嫩，微帶黏液感",
      "notes": "不建議生食。可涼拌（川燙後）或炒肉絲。"
    },
    "nutrition": "蛋白質、鐵質、維生素C",
    "caution": "避免採食不認識的蕨類。部分蕨類含致癌物質，過貓需煮熟食用。",
    "confusion_risk": "medium",
    "distribution": {
      "altitude_range": [0, 1000],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  },
  {
    "id": "hedychium_coronarium",
    "scientific_name": "Hedychium coronarium",
    "common_names": { "zh-TW": "野薑花", "en": "White Ginger Lily" },
    "category": "edible_and_medicinal",
    "edible_parts": ["花瓣", "嫩莖", "根莖（調味用）"],
    "morphology": {
      "leaf_shape": "長橢圓形披針狀，長 20-40cm，互生排列",
      "leaf_surface": "綠色，光滑，葉背微被毛",
      "flower": "白色蝴蝶形花朵，香氣濃郁（關鍵辨識特徵）",
      "stem": "直立草本，高 100-200cm",
      "rhizome": "根莖塊狀，有薑味"
    },
    "habitat": "溪邊、水田旁、潮濕地，海拔 0-800m",
    "harvesting": {
      "target": "花朵（盛開時最佳）、嫩莖心",
      "method": "花朵用手摘取。嫩莖從基部折斷。",
      "season": "花期 6-10 月",
      "sustainability": "採花不影響植株生長"
    },
    "preparation": {
      "method": "花瓣可直接食用或入菜。嫩莖可炒食。根莖可當薑使用。",
      "cooking_time": "嫩莖快炒 3-5 分鐘",
      "taste": "花瓣清香，嫩莖清脆，根莖辛辣如薑",
      "notes": "野薑花粽是台灣知名料理。花朵可泡茶。"
    },
    "nutrition": "維生素C、礦物質",
    "medicinal_uses": ["根莖：去寒、消腫"],
    "caution": "確認花朵為純白蝴蝶形且有濃郁香氣。勿與其他薑科植物混淆。",
    "confusion_risk": "medium",
    "distribution": {
      "altitude_range": [0, 800],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  },
  {
    "id": "plantago_major",
    "scientific_name": "Plantago major",
    "common_names": { "zh-TW": "車前草", "en": "Broadleaf Plantain" },
    "category": "edible_and_medicinal",
    "edible_parts": ["嫩葉"],
    "morphology": {
      "leaf_shape": "橢圓形至卵形，長 5-20cm，寬 3-10cm",
      "leaf_arrangement": "基生蓮座狀（所有葉從基部長出，貼地展開）",
      "leaf_surface": "綠色，葉面有明顯的 3-7 條平行脈（關鍵特徵）",
      "leaf_margin": "全緣或微波狀",
      "flower": "穗狀花序，直立花莖，長 5-20cm，小花密集",
      "stem": "無明顯莖，僅有花莖直立"
    },
    "habitat": "路邊、荒地、草地、林緣、步道旁，海拔 0-2500m，極常見",
    "harvesting": {
      "target": "嫩葉（春季最嫩）",
      "method": "從基部摘取嫩葉",
      "season": "春夏最佳",
      "sustainability": "保留根部，會持續長出新葉"
    },
    "preparation": {
      "method": "可生食或煮食。嫩葉川燙後涼拌。",
      "cooking_time": "川燙 1-2 分鐘",
      "taste": "略帶苦味，嫩葉較甜",
      "notes": "老葉纖維多，較適合煮湯。"
    },
    "nutrition": "維生素A、C、K、鈣、鐵",
    "medicinal_uses": ["消炎", "抗菌", "止血", "利尿", "搗碎鮮葉敷傷口可止血消炎"],
    "caution": "確認平行脈特徵。避免採集路邊受農藥或汽車廢氣污染的植株。",
    "confusion_risk": "low",
    "distribution": {
      "altitude_range": [0, 2500],
      "climate_zones": ["亞熱帶", "溫帶", "熱帶"]
    }
  },
  {
    "id": "solanum_nigrum",
    "scientific_name": "Solanum nigrum",
    "common_names": { "zh-TW": "龍葵（黑籽仔菜）", "en": "Black Nightshade" },
    "category": "edible",
    "edible_parts": ["嫩莖葉", "成熟果實（全黑時）"],
    "morphology": {
      "leaf_shape": "卵形至橢圓形，長 3-10cm，葉緣全緣或微波狀",
      "leaf_surface": "綠色，薄紙質",
      "flower": "白色小花，5 瓣，花心黃色，2-8 朵成聚繖花序",
      "fruit": "球形漿果，直徑 6-8mm，未熟時綠色，成熟後全黑（關鍵）",
      "stem": "草本，高 30-100cm，莖多分枝"
    },
    "habitat": "荒地、田邊、路旁、廢耕地，海拔 0-1500m，極常見",
    "harvesting": {
      "target": "嫩莖葉（全年）。果實僅採全黑成熟者。",
      "method": "摘取嫩枝頂端 10-15cm",
      "season": "全年可採",
      "sustainability": "摘取嫩枝會促進分枝生長"
    },
    "preparation": {
      "method": "嫩莖葉川燙或炒食。成熟黑果可直接食用。",
      "cooking_time": "川燙 2-3 分鐘",
      "taste": "嫩葉略帶苦味，黑果微甜",
      "notes": "是台灣常見的野菜，常用蒜末快炒。"
    },
    "nutrition": "維生素C、蛋白質、鈣",
    "caution": "⚠️ 未成熟的綠色果實含龍葵鹼（solanine），有毒！只能食用完全成熟的黑色果實。嫩莖葉需煮熟。",
    "confusion_risk": "medium",
    "distribution": {
      "altitude_range": [0, 1500],
      "climate_zones": ["亞熱帶", "溫帶", "熱帶"]
    }
  },
  {
    "id": "crassocephalum_crepidioides",
    "scientific_name": "Crassocephalum crepidioides",
    "common_names": { "zh-TW": "昭和草（飛機草）", "en": "Redflower Ragleaf" },
    "category": "edible",
    "edible_parts": ["嫩莖葉"],
    "morphology": {
      "leaf_shape": "長橢圓形，長 5-15cm，葉緣有不規則鋸齒或裂片",
      "leaf_surface": "綠色，略帶紫暈，柔軟",
      "flower": "頭狀花序，下垂，花冠橘紅色（關鍵特徵）",
      "stem": "直立草本，高 20-80cm，莖中空，斷面有白色乳汁",
      "fruit": "瘦果帶白色冠毛（似蒲公英），可隨風飄散"
    },
    "habitat": "荒地、路旁、農田邊、開闊地，海拔 0-2000m，非常常見",
    "harvesting": {
      "target": "嫩莖葉（開花前最嫩最甜）",
      "method": "摘取頂端嫩莖 10-20cm",
      "season": "全年可採，冬春最佳",
      "sustainability": "生長快速，採集不影響族群"
    },
    "preparation": {
      "method": "川燙後涼拌、炒食、煮湯皆可。",
      "cooking_time": "川燙 1-2 分鐘",
      "taste": "口感滑嫩，有特殊清香，略帶甘甜",
      "notes": "味道類似茼蒿，是野外最容易取得的野菜之一。可煮味噌湯。"
    },
    "nutrition": "維生素A、C、鈣、鐵",
    "caution": "辨識重點是橘紅色下垂花序和斷莖白色乳汁。避免與其他菊科植物混淆。",
    "confusion_risk": "low",
    "distribution": {
      "altitude_range": [0, 2000],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  },
  {
    "id": "colocasia_esculenta",
    "scientific_name": "Colocasia esculenta",
    "common_names": { "zh-TW": "芋頭", "en": "Taro" },
    "category": "edible",
    "edible_parts": ["塊莖（須煮熟）", "嫩莖（須煮熟）"],
    "morphology": {
      "leaf_shape": "心形，長 20-50cm",
      "leaf_surface": "有天鵝絨般的微絨毛質感，不太光滑",
      "water_droplet_test": "水珠在葉面呈圓珠狀滾動（荷葉效應）",
      "leaf_tip": "通常朝下垂",
      "petiole": "接在葉片邊緣內側（盾狀著生），綠色，較少紫暈",
      "underground": "有圓形塊莖"
    },
    "habitat": "田間栽培、溪邊野生化，海拔 0-1000m",
    "harvesting": {
      "target": "塊莖（地下部分）",
      "method": "挖掘地下塊莖",
      "season": "全年可採，秋冬塊莖最大",
      "sustainability": "保留部分塊莖可再生"
    },
    "preparation": {
      "method": "必須完全煮熟！生芋頭含草酸鈣，食用會導致口腔麻痺。",
      "cooking_time": "蒸煮 20-30 分鐘至熟透",
      "taste": "鬆軟綿密，微甜",
      "notes": "處理生芋頭時建議戴手套，汁液可能引起皮膚搔癢。"
    },
    "nutrition": "碳水化合物（能量來源）、膳食纖維、鉀",
    "caution": "⚠️ 極易與有毒的姑婆芋混淆！必須做水珠測試和葉柄接合處確認。若無法確定，禁止食用。生芋頭不可食用。",
    "confusion_risk": "critical",
    "distribution": {
      "altitude_range": [0, 1000],
      "climate_zones": ["亞熱帶", "熱帶"]
    }
  },
  {
    "id": "auricularia_auricula_judae",
    "scientific_name": "Auricularia auricula-judae",
    "common_names": { "zh-TW": "木耳", "en": "Wood Ear Mushroom" },
    "category": "edible",
    "edible_parts": ["子實體（整朵）"],
    "morphology": {
      "shape": "耳狀或碗狀，直徑 3-10cm",
      "surface": "外面有細微絨毛，灰褐色。內面光滑，深褐色至黑色。",
      "texture": "新鮮時膠質狀、半透明、有彈性。乾燥後硬縮，泡水恢復。",
      "growth_pattern": "叢生於枯木或腐木表面（關鍵特徵）",
      "color": "深褐色至黑色"
    },
    "habitat": "枯木、腐木表面，潮濕森林中，海拔 0-2000m",
    "harvesting": {
      "target": "新鮮、完整、無異味的子實體",
      "method": "用手或刀從木頭上摘取",
      "season": "雨後最多，春秋最佳",
      "sustainability": "保留部分讓其釋放孢子繁殖"
    },
    "preparation": {
      "method": "川燙或炒食。新鮮木耳可直接料理。",
      "cooking_time": "川燙 2-3 分鐘或快炒 3-5 分鐘",
      "taste": "口感脆嫩滑彈，幾乎無味，吸附湯汁佳",
      "notes": "涼拌時建議先川燙殺菌。"
    },
    "nutrition": "鐵質、膳食纖維、多醣體",
    "caution": "確認生長在枯木上（而非土壤）。避免採集已腐爛、發臭、或顏色異常的菇體。不確定時禁止食用。",
    "confusion_risk": "low",
    "distribution": {
      "altitude_range": [0, 2000],
      "climate_zones": ["亞熱帶", "溫帶", "熱帶"]
    }
  },
  {
    "id": "nephrolepis_cordifolia",
    "scientific_name": "Nephrolepis cordifolia",
    "common_names": { "zh-TW": "腎蕨（山鳳梨）", "en": "Tuberous Sword Fern" },
    "category": "edible_and_water",
    "edible_parts": ["球形儲水塊莖（地下）"],
    "morphology": {
      "leaf_shape": "一回羽狀複葉，長 30-80cm，小葉互生，長 2-4cm",
      "leaf_arrangement": "叢生，向外弧形展開",
      "leaf_surface": "綠色，薄草質",
      "underground": "根部有球形儲水塊莖，直徑 1-2cm（關鍵特徵，含水分）",
      "growth_pattern": "地生或附生，常見於路旁岩壁"
    },
    "habitat": "低中海拔林緣、路旁、岩壁，海拔 0-1500m，極常見",
    "harvesting": {
      "target": "地下球形儲水塊莖",
      "method": "挖掘根部，摘取球形塊莖，剝去外皮",
      "season": "全年可採",
      "sustainability": "保留根系，會再生塊莖"
    },
    "preparation": {
      "method": "剝皮後可直接生食或煮食。塊莖含水分，可解渴。",
      "cooking_time": "可生食，或煮 5-10 分鐘",
      "taste": "微甜，含水分，口感像荸薺",
      "notes": "在缺水情況下是重要的水分來源。台灣原住民稱為「山鳳梨」。"
    },
    "nutrition": "水分（主要價值）、碳水化合物",
    "caution": "確認是腎蕨（一回羽狀複葉 + 地下球形塊莖）。其他蕨類無此塊莖。",
    "confusion_risk": "low",
    "distribution": {
      "altitude_range": [0, 1500],
      "climate_zones": ["亞熱帶", "熱帶", "溫帶"]
    }
  },
  {
    "id": "oenanthe_javanica",
    "scientific_name": "Oenanthe javanica",
    "common_names": { "zh-TW": "山芹菜（水芹菜）", "en": "Water Dropwort" },
    "category": "edible",
    "edible_parts": ["嫩莖葉"],
    "morphology": {
      "leaf_shape": "二至三回羽狀複葉，小葉卵形至菱形，長 1-3cm，葉緣鋸齒狀",
      "leaf_surface": "綠色，光滑",
      "flower": "複繖形花序，白色小花",
      "stem": "中空，光滑，有節，匍匐莖可水生",
      "smell": "有獨特的芹菜香味（關鍵辨識特徵）"
    },
    "habitat": "溪邊、水田、潮濕地、沼澤邊緣，海拔 0-1500m",
    "harvesting": {
      "target": "嫩莖葉（頂端 10-15cm）",
      "method": "從節上方折取嫩莖",
      "season": "全年可採，春季最嫩",
      "sustainability": "匍匐莖會持續生長"
    },
    "preparation": {
      "method": "川燙或快炒。香味濃郁，適合炒肉絲或煮湯。",
      "cooking_time": "快炒 2-3 分鐘",
      "taste": "芹菜香味濃郁，口感脆嫩",
      "notes": "味道比一般芹菜更濃郁。"
    },
    "nutrition": "維生素A、C、鈣、鐵",
    "caution": "⚠️ 注意！同屬的毒水芹（Oenanthe crocata）劇毒。確認有芹菜香味，且生長在乾淨水源旁。水質不佳的地方不建議採集（寄生蟲風險）。",
    "confusion_risk": "high",
    "distribution": {
      "altitude_range": [0, 1500],
      "climate_zones": ["亞熱帶", "溫帶"]
    }
  }
]
'''
EDIBLE_PLANTS = json.loads(_EDIBLE_PLANTS_JSON)

_DANGEROUS_ANIMALS_JSON = r'''
[
  {
    "id": "trimeresurus_stejnegeri",
    "scientific_name": "Trimeresurus stejnegeri",
    "common_names": { "zh-TW": "赤尾青竹絲", "en": "Stejneger's Green Bamboo Viper" },
    "category": "venomous_snake",
    "danger_level": "high",
    "venom_type": "出血性毒",
    "morphology": {
      "body_color": "全身鮮綠色，體側有白色或淡黃色縱線",
      "tail": "尾巴末端為磚紅色（關鍵特徵）",
      "head_shape": "三角形，明顯比頸部寬（蝮蛇科特徵）",
      "pupil": "縱裂瞳孔（貓眼狀）",
      "pit_organ": "眼與鼻孔之間有明顯的頰窩（紅外線感應器）",
      "body_size": "中小型，全長 60-90cm，體型細長",
      "scales": "背鱗有稜脊"
    },
    "habitat": "低中海拔森林、竹林、灌木叢，常纏繞在樹枝上，海拔 0-2000m",
    "behavior": "夜行性，攻擊性中等。常靜止不動偽裝在枝葉間。受驚時會快速咬擊。",
    "bite_symptoms": [
      "傷口劇烈疼痛",
      "局部迅速腫脹",
      "出血不止（凝血功能障礙）",
      "傷口周圍瘀青",
      "嚴重者可能組織壞死"
    ],
    "first_aid": {
      "do": [
        "保持冷靜，減少活動",
        "記住蛇的外觀特徵（拍照更佳）",
        "移除傷肢上的戒指、手錶等束縛物",
        "保持傷肢低於心臟位置",
        "儘速送醫，需施打抗蛇毒血清"
      ],
      "do_not": [
        "勿切開傷口",
        "勿用嘴吸毒",
        "勿綁止血帶（加壓繃帶例外）",
        "勿冰敷",
        "勿飲酒",
        "勿嘗試捕蛇"
      ],
      "antivenom": "出血性蛇毒血清（台灣各大醫院急診備有）"
    },
    "activity_pattern": "夜行性為主，陰天或雨後白天也會活動",
    "distribution": {
      "altitude_range": [0, 2000],
      "climate_zones": ["亞熱帶", "熱帶"]
    },
    "encounter_frequency": "台灣最常見的毒蛇，咬傷案例數最多"
  },
  {
    "id": "protobothrops_mucrosquamatus",
    "scientific_name": "Protobothrops mucrosquamatus",
    "common_names": { "zh-TW": "龜殼花", "en": "Taiwan Habu" },
    "category": "venomous_snake",
    "danger_level": "high",
    "venom_type": "出血性毒",
    "morphology": {
      "body_color": "灰褐色底，背部有深褐色大型龜甲狀斑紋（關鍵特徵）",
      "head_shape": "三角形，寬大，與頸部分界明顯",
      "pupil": "縱裂瞳孔",
      "pit_organ": "眼前方有頰窩",
      "body_size": "中大型，全長 80-130cm，體型粗壯",
      "scales": "背鱗有強稜脊，體表粗糙感"
    },
    "habitat": "低海拔森林、農田附近、住家周圍、石堆、柴堆，海拔 0-1500m",
    "behavior": "夜行性，攻擊性較高。受驚時會快速反擊。常出現在人類活動區域附近。",
    "bite_symptoms": [
      "傷口劇烈腫痛",
      "迅速腫脹擴散",
      "出血不止",
      "組織壞死（嚴重者）",
      "可能需要截肢（極嚴重者）"
    ],
    "first_aid": {
      "do": [
        "保持冷靜，減少活動",
        "記住蛇的外觀特徵",
        "移除傷肢束縛物",
        "保持傷肢低於心臟位置",
        "儘速送醫"
      ],
      "do_not": [
        "勿切開傷口",
        "勿用嘴吸毒",
        "勿綁止血帶",
        "勿冰敷",
        "勿嘗試捕蛇"
      ],
      "antivenom": "出血性蛇毒血清"
    },
    "activity_pattern": "夜行性，黃昏至夜間最活躍",
    "distribution": {
      "altitude_range": [0, 1500],
      "climate_zones": ["亞熱帶"]
    },
    "encounter_frequency": "台灣第二常見的毒蛇咬傷來源"
  },
  {
    "id": "bungarus_multicinctus",
    "scientific_name": "Bungarus multicinctus",
    "common_names": { "zh-TW": "雨傘節（銀環蛇）", "en": "Many-banded Krait" },
    "category": "venomous_snake",
    "danger_level": "critical",
    "venom_type": "神經性毒",
    "morphology": {
      "body_color": "全身黑白相間的環狀橫帶（關鍵特徵），黑白帶等寬或黑帶略寬",
      "head_shape": "橢圓形，與頸部區分不明顯（非三角形）",
      "pupil": "圓形瞳孔",
      "body_size": "中型，全長 100-150cm",
      "dorsal_scales": "背部中央一列鱗片明顯擴大呈六角形（關鍵特徵）",
      "tail": "尾部短，末端鈍圓"
    },
    "habitat": "低海拔農田、水邊、住家附近、排水溝，海拔 0-500m",
    "behavior": "夜行性，動作緩慢，白天溫馴。但毒性極強！夜間會主動覓食。",
    "bite_symptoms": [
      "⚠️ 咬傷初期幾乎不痛（極度危險！容易被忽視）",
      "數小時後出現眼瞼下垂",
      "吞嚥困難",
      "全身肌肉無力",
      "呼吸困難",
      "呼吸衰竭（可致死）"
    ],
    "first_aid": {
      "do": [
        "⚠️ 即使不痛也必須立即就醫！這是最危險的部分",
        "保持冷靜",
        "記住蛇的外觀（黑白相間）",
        "可使用彈性繃帶加壓包紮（壓力固定法），減緩毒液擴散",
        "儘速送醫，需施打神經性蛇毒血清",
        "注意呼吸狀況，準備人工呼吸"
      ],
      "do_not": [
        "勿忽視！即使傷口不痛也要就醫",
        "勿切開傷口",
        "勿用嘴吸毒",
        "勿綁止血帶"
      ],
      "antivenom": "神經性蛇毒血清（時間是關鍵，越早施打越好）"
    },
    "activity_pattern": "嚴格夜行性",
    "distribution": {
      "altitude_range": [0, 500],
      "climate_zones": ["亞熱帶"]
    },
    "encounter_frequency": "不常見，但致死率最高的台灣毒蛇"
  },
  {
    "id": "naja_atra",
    "scientific_name": "Naja atra",
    "common_names": { "zh-TW": "眼鏡蛇（飯匙倩）", "en": "Chinese Cobra" },
    "category": "venomous_snake",
    "danger_level": "critical",
    "venom_type": "混合毒（神經毒 + 細胞毒）",
    "morphology": {
      "body_color": "全身黑色或深褐色，腹面淡色",
      "hood": "受威脅時頸部張開呈扁平狀（關鍵特徵），可見白色眼鏡狀斑紋",
      "head_shape": "橢圓形，受威脅時頸部明顯擴展",
      "pupil": "圓形瞳孔",
      "body_size": "中大型，全長 120-150cm",
      "scales": "背鱗光滑"
    },
    "habitat": "低海拔農田、溪邊、竹林、住家附近，海拔 0-800m",
    "behavior": "日行性為主。受驚時會抬起前身、張開頸部、發出嘶嘶聲警告。可噴毒（射程達 2m）。",
    "bite_symptoms": [
      "傷口疼痛",
      "局部腫脹",
      "組織壞死",
      "眼瞼下垂（神經毒症狀）",
      "呼吸困難",
      "被噴毒液入眼：劇烈疼痛、視力受損"
    ],
    "first_aid": {
      "do": [
        "立即遠離（此蛇攻擊範圍大）",
        "儘速送醫",
        "如毒液噴入眼睛：立即用大量清水沖洗至少 15 分鐘",
        "記住蛇的外觀特徵"
      ],
      "do_not": [
        "勿嘗試與其對峙",
        "勿切開傷口",
        "勿用嘴吸毒"
      ],
      "antivenom": "神經性 + 出血性混合蛇毒血清"
    },
    "activity_pattern": "日行性為主，偶爾夜間活動",
    "distribution": {
      "altitude_range": [0, 800],
      "climate_zones": ["亞熱帶"]
    },
    "encounter_frequency": "中等，但攻擊性強，咬傷後果嚴重"
  },
  {
    "id": "deinagkistrodon_acutus",
    "scientific_name": "Deinagkistrodon acutus",
    "common_names": { "zh-TW": "百步蛇（尖吻蝮）", "en": "Hundred-pace Pit Viper" },
    "category": "venomous_snake",
    "danger_level": "critical",
    "venom_type": "出血性毒 + 細胞毒",
    "morphology": {
      "body_color": "灰褐色底，背部有大型深色三角形斑紋，排列成鏈狀",
      "head_shape": "大三角形，吻端明顯上翹（關鍵特徵）",
      "pupil": "縱裂瞳孔",
      "pit_organ": "眼前方有明顯頰窩",
      "body_size": "大型，全長 120-150cm，體型極粗壯",
      "tail": "尾端有角質化尖刺"
    },
    "habitat": "中低海拔原始森林、山區，岩石堆、落葉堆中，海拔 300-1500m",
    "behavior": "動作遲緩，但攻擊速度極快。保護色極佳，常隱藏在落葉中不易發現。台灣原住民族的文化圖騰。",
    "bite_symptoms": [
      "傷口劇烈疼痛",
      "迅速大面積腫脹",
      "大量出血",
      "嚴重組織壞死",
      "全身性凝血功能障礙",
      "可致死或截肢"
    ],
    "first_aid": {
      "do": [
        "保持冷靜，立即遠離",
        "減少活動，勿奔跑",
        "記住蛇的特徵（三角形頭、上翹吻端）",
        "儘速送醫，此為急重症",
        "可用彈性繃帶加壓包紮"
      ],
      "do_not": [
        "勿切開傷口",
        "勿用嘴吸毒",
        "勿綁止血帶",
        "勿嘗試捕蛇（此蛇為保育類）"
      ],
      "antivenom": "出血性蛇毒血清（大劑量）"
    },
    "activity_pattern": "夜行性為主，白天在落葉中靜臥",
    "distribution": {
      "altitude_range": [300, 1500],
      "climate_zones": ["亞熱帶"]
    },
    "encounter_frequency": "少見（保育類），但咬傷後果極嚴重"
  }
]
'''
DANGEROUS_ANIMALS = json.loads(_DANGEROUS_ANIMALS_JSON)

_SNAKEBITE_FIRST_AID_JSON = r'''
{
  "scenario": "snakebite",
  "title_zh": "蛇咬傷緊急處理 SOP",
  "steps": [
    {
      "step": 1,
      "action_zh": "保持冷靜，立即遠離蛇的攻擊範圍（至少 2 公尺以上）",
      "reason_zh": "恐慌會加速心跳，使毒液擴散更快"
    },
    {
      "step": 2,
      "action_zh": "記住蛇的外觀特徵：體色、斑紋、頭形、大小。拍照更佳。",
      "reason_zh": "醫院需要判斷蛇種以選擇正確的抗蛇毒血清"
    },
    {
      "step": 3,
      "action_zh": "移除傷肢上的戒指、手錶、手環等束縛物",
      "reason_zh": "腫脹會使這些物品卡住，造成血流阻斷"
    },
    {
      "step": 4,
      "action_zh": "保持傷肢低於心臟位置，減少活動",
      "reason_zh": "減緩毒液經由淋巴和血液系統擴散"
    },
    {
      "step": 5,
      "action_zh": "如有彈性繃帶，可從傷口上方（靠近心臟方向）開始包紮，壓力與包紮扭傷的力度相同",
      "reason_zh": "壓力固定法（Pressure Immobilization）可減緩神經性毒液擴散，但對出血性毒液效果有限"
    },
    {
      "step": 6,
      "action_zh": "儘速送醫或呼叫救援",
      "reason_zh": "抗蛇毒血清是唯一有效的治療方式"
    }
  ],
  "do_not": [
    "❌ 勿切開傷口放血",
    "❌ 勿用嘴吸毒（口腔黏膜會吸收毒液）",
    "❌ 勿綁止血帶（可能導致組織壞死）",
    "❌ 勿冰敷（加速組織損傷）",
    "❌ 勿飲酒（加速血液循環）",
    "❌ 勿自行服藥",
    "❌ 勿嘗試捕蛇（可能造成二次咬傷）"
  ],
  "by_venom_type": {
    "hemorrhagic": {
      "name_zh": "出血性毒（赤尾青竹絲、龜殼花、百步蛇）",
      "symptoms_zh": "傷口劇痛、迅速腫脹、出血不止、瘀青",
      "key_action_zh": "減少活動，勿加壓包紮（可能加重局部出血）",
      "antivenom_zh": "出血性蛇毒血清"
    },
    "neurotoxic": {
      "name_zh": "神經性毒（雨傘節）",
      "symptoms_zh": "⚠️ 初期可能不痛！之後眼瞼下垂、吞嚥困難、呼吸困難",
      "key_action_zh": "即使不痛也必須就醫！可用壓力固定法。注意呼吸，準備人工呼吸。",
      "antivenom_zh": "神經性蛇毒血清（越早施打越好）"
    },
    "mixed": {
      "name_zh": "混合毒（眼鏡蛇）",
      "symptoms_zh": "局部疼痛腫脹 + 神經症狀",
      "key_action_zh": "按出血性處理，同時注意呼吸狀況",
      "antivenom_zh": "混合蛇毒血清"
    }
  }
}
'''
SNAKEBITE_FIRST_AID = json.loads(_SNAKEBITE_FIRST_AID_JSON)

_PLANT_POISONING_FIRST_AID_JSON = r'''
{
  "scenario": "plant_poisoning",
  "title_zh": "植物中毒緊急處理 SOP",
  "general_steps": [
    {
      "step": 1,
      "action_zh": "立即停止食用，將口中殘餘物吐出",
      "reason_zh": "減少毒素攝入量"
    },
    {
      "step": 2,
      "action_zh": "用大量清水漱口（勿吞嚥）",
      "reason_zh": "沖洗口腔中的毒素"
    },
    {
      "step": 3,
      "action_zh": "保留植物樣本或拍照記錄",
      "reason_zh": "醫師需要辨識植物種類以決定治療方案"
    },
    {
      "step": 4,
      "action_zh": "若意識清醒且在食後 1 小時內，可考慮催吐（用手指觸碰喉嚨後方）",
      "reason_zh": "減少毒素吸收。但某些情況不適用，見下方禁忌。"
    },
    {
      "step": 5,
      "action_zh": "補充飲水（小口慢喝清水）",
      "reason_zh": "稀釋胃中毒素"
    },
    {
      "step": 6,
      "action_zh": "儘速送醫或呼叫救援",
      "reason_zh": "部分植物毒素有延遲性（如鵝膏毒素），初期無症狀不代表安全"
    }
  ],
  "do_not_induce_vomiting_when": [
    "患者意識不清或昏迷",
    "患者有抽搐現象",
    "食入腐蝕性植物汁液（如姑婆芋——催吐會二次灼傷食道）",
    "食後已超過 2 小時（毒素已進入腸道）"
  ],
  "by_toxin_type": {
    "oxalate_crystal": {
      "name_zh": "草酸鈣針晶中毒（姑婆芋等天南星科）",
      "symptoms_zh": "口腔灼痛、舌頭麻木腫脹、流涎、喉嚨腫脹、吞嚥困難",
      "first_aid_zh": [
        "勿催吐（避免二次灼傷食道）",
        "大量清水漱口",
        "可含冰塊或喝冰牛奶緩解口腔疼痛",
        "注意呼吸道——如喉嚨嚴重腫脹可能影響呼吸",
        "儘速就醫"
      ]
    },
    "cardiac_glycoside": {
      "name_zh": "強心苷中毒（夾竹桃、海檬果）",
      "symptoms_zh": "噁心嘔吐、心律不整、心跳過慢、視覺異常（看到黃色光暈）",
      "first_aid_zh": [
        "立即催吐（若意識清醒且在 1 小時內）",
        "儘速送醫——此為心臟急症",
        "需持續心電圖監測",
        "告知醫師疑似強心苷中毒"
      ]
    },
    "tropane_alkaloid": {
      "name_zh": "莨菪鹼中毒（曼陀羅）",
      "symptoms_zh": "瞳孔放大、口乾、皮膚潮紅、心跳加速、幻覺、譫妄",
      "first_aid_zh": [
        "催吐（若意識清醒且在 1 小時內）",
        "保持患者在安全環境中（幻覺可能導致危險行為）",
        "維持呼吸道暢通",
        "儘速送醫"
      ]
    },
    "amatoxin": {
      "name_zh": "鵝膏毒素中毒（鱗柄白毒鵝膏等）",
      "symptoms_zh": "⚠️ 潛伏期 6-24 小時！初期嘔吐腹瀉 → 假性恢復 → 肝腎衰竭",
      "first_aid_zh": [
        "⚠️ 食用不明菇類後即使無症狀也必須就醫",
        "催吐（若在食後短時間內）",
        "保留菇體樣本送醫",
        "告知醫師疑似鵝膏毒素中毒",
        "此毒素無特效解毒劑，需緊急支持性治療",
        "時間是關鍵——越早治療存活率越高"
      ]
    }
  },
  "skin_contact": {
    "title_zh": "皮膚接觸有毒植物處理",
    "steps": [
      "立即用大量清水沖洗接觸部位（15 分鐘以上）",
      "如有肥皂，用肥皂水清洗（對漆樹類特別有效，需在 15 分鐘內）",
      "被焮刺毛刺傷（咬人貓/咬人狗）：用膠帶反覆黏除皮膚上的刺毛",
      "勿搔抓患處",
      "可塗抹含類固醇的止癢藥膏",
      "嚴重腫脹或過敏反應（呼吸困難、全身蕁麻疹）：立即就醫"
    ]
  }
}
'''
PLANT_POISONING_FIRST_AID = json.loads(_PLANT_POISONING_FIRST_AID_JSON)

_WOUND_CARE_JSON = r'''
{
  "scenario": "wound_care",
  "title_zh": "野外傷口處理 SOP",
  "wound_types": {
    "cut_abrasion": {
      "name_zh": "切割傷 / 擦傷",
      "steps": [
        "用清水（飲用水）沖洗傷口，去除泥土和碎屑",
        "如有出血，用乾淨布料加壓止血 10-15 分鐘",
        "傷口乾淨後，可用車前草葉搗碎敷在傷口上（天然止血消炎）",
        "用乾淨布料或大型葉片包紮傷口",
        "每 4-6 小時更換敷料"
      ],
      "natural_remedies": [
        {
          "plant_id": "plantago_major",
          "name_zh": "車前草",
          "usage_zh": "搗碎新鮮葉片，敷在傷口上，有止血、消炎、抗菌效果",
          "dosage_zh": "5-6 片新鮮葉片搗碎成泥，每 4-6 小時更換"
        }
      ]
    },
    "burn": {
      "name_zh": "燒燙傷",
      "steps": [
        "立即用大量清水沖洗患處至少 20 分鐘",
        "移除患處的衣物和飾品（如尚未黏住皮膚）",
        "勿刺破水泡",
        "用乾淨濕布覆蓋患處",
        "嚴重燒傷（大面積、三度燒傷）需儘速就醫"
      ],
      "natural_remedies": []
    },
    "sprain": {
      "name_zh": "扭傷",
      "steps": [
        "停止活動，休息（Rest）",
        "如有清潔冷水，可冷敷患處 15-20 分鐘（Ice）",
        "用布條或彈性材料包紮固定（Compression）",
        "盡量抬高患肢（Elevation）",
        "R.I.C.E. 原則"
      ],
      "natural_remedies": []
    },
    "insect_sting": {
      "name_zh": "蜂螫 / 蟲咬",
      "steps": [
        "遠離蜂巢區域",
        "如有螫針殘留，用硬物（如信用卡邊緣）刮除，勿用手指擠壓",
        "用清水沖洗患處",
        "冷敷減緩腫脹",
        "⚠️ 注意過敏反應：呼吸困難、全身蕁麻疹、頭暈 → 可能是過敏性休克，立即就醫"
      ],
      "natural_remedies": [
        {
          "plant_id": "plantago_major",
          "name_zh": "車前草",
          "usage_zh": "搗碎葉片敷在蟲咬處，可減輕腫脹和搔癢",
          "dosage_zh": "數片葉片搗碎敷上"
        }
      ]
    },
    "dehydration": {
      "name_zh": "脫水",
      "steps": [
        "尋找水源（溪水、雨水、植物儲水）",
        "如有火源，將水煮沸至少 1 分鐘再飲用",
        "無法煮沸時，優先選擇流動的溪水而非靜止的水塘",
        "腎蕨的地下儲水塊莖可提供少量水分",
        "小口慢飲，避免一次大量飲水導致嘔吐"
      ],
      "natural_remedies": [
        {
          "plant_id": "nephrolepis_cordifolia",
          "name_zh": "腎蕨（山鳳梨）",
          "usage_zh": "挖出地下球形儲水塊莖，剝皮後可直接食用，含有水分",
          "dosage_zh": "多顆塊莖可提供少量水分應急"
        }
      ]
    }
  }
}
'''
WOUND_CARE = json.loads(_WOUND_CARE_JSON)

print(f"✅ 知識庫載入完成：有毒植物 {len(TOXIC_PLANTS)} 種、混淆物種 {len(CONFUSION_PAIRS)} 對、可食植物 {len(EDIBLE_PLANTS)} 種、危險動物 {len(DANGEROUS_ANIMALS)} 種")


## 核心模組

Context Engine、Prompt Builder、Response Parser、Model Loader、Pipeline

In [ ]:
# Cell 6: Context Engine (環境上下文引擎)

@dataclass
class EnvironmentContext:
    region_id: str = DEFAULT_REGION
    country: str = "台灣"
    climate_zone: str = "亞熱帶"
    altitude: int = 500
    vegetation_zone: str = "低海拔闊葉林"
    season: str = ""
    month: int = 0

    def __post_init__(self):
        if not self.month:
            self.month = datetime.now().month
        if not self.season:
            self.season = self._month_to_season(self.month)

    @staticmethod
    def _month_to_season(month: int) -> str:
        if month in (3, 4, 5):
            return "春季"
        elif month in (6, 7, 8):
            return "夏季"
        elif month in (9, 10, 11):
            return "秋季"
        else:
            return "冬季"

    def to_prompt_header(self) -> str:
        return (
            f"【環境上下文】\n"
            f"地點：{self.country}\n"
            f"氣候帶：{self.climate_zone}\n"
            f"海拔：約 {self.altitude}m\n"
            f"植被帶：{self.vegetation_zone}\n"
            f"季節：{self.season}"
        )


@dataclass
class KnowledgeBase:
    toxic_plants: list = field(default_factory=list)
    confusion_pairs: list = field(default_factory=list)
    edible_plants: list = field(default_factory=list)
    dangerous_animals: list = field(default_factory=list)
    snakebite_first_aid: dict = field(default_factory=dict)
    plant_poisoning_first_aid: dict = field(default_factory=dict)
    wound_care: dict = field(default_factory=dict)


def load_knowledge_base_embedded() -> KnowledgeBase:
    """從嵌入的全域變數載入知識庫（Kaggle Notebook 用）"""
    return KnowledgeBase(
        toxic_plants=TOXIC_PLANTS,
        confusion_pairs=CONFUSION_PAIRS,
        edible_plants=EDIBLE_PLANTS,
        dangerous_animals=DANGEROUS_ANIMALS,
        snakebite_first_aid=SNAKEBITE_FIRST_AID,
        plant_poisoning_first_aid=PLANT_POISONING_FIRST_AID,
        wound_care=WOUND_CARE,
    )


def create_context(
    country="台灣", altitude=500, climate_zone="亞熱帶",
    vegetation_zone="低海拔闊葉林", region_id=DEFAULT_REGION,
) -> EnvironmentContext:
    return EnvironmentContext(
        region_id=region_id, country=country,
        climate_zone=climate_zone, altitude=altitude,
        vegetation_zone=vegetation_zone,
    )


# 測試
ctx = create_context()
kb = load_knowledge_base_embedded()
print(ctx.to_prompt_header())
print(f"\n✅ Context Engine 載入完成，知識庫：{len(kb.toxic_plants)} 有毒植物、{len(kb.confusion_pairs)} 混淆對")

In [ ]:
# Cell 7: Prompt Builder (動態 Prompt 組裝器)

def _format_toxic_plant(plant: dict) -> str:
    morph = plant.get("morphology", {})
    lines = [f"{plant['common_names']['zh-TW']} ({plant['scientific_name']})"]
    lines.append(f"   毒性：{plant['toxicity']}")
    field_map = {
        "leaf_shape": "葉形", "leaf_surface": "葉面", "leaf_venation": "葉脈",
        "leaf_tip": "葉尖", "petiole": "葉柄", "stem": "莖部",
        "flower": "花", "fruit": "果實", "leaf_arrangement": "葉排列",
        "cap": "傘蓋", "gills": "菌褶", "stipe": "菌柄",
        "spore_print": "孢子印", "ring": "菌環", "flesh": "菌肉",
    }
    for key, label in field_map.items():
        if key in morph:
            lines.append(f"   {label}：{morph[key]}")
    lines.append(f"   生長環境：{plant.get('habitat', 'N/A')}")
    if plant.get("confusion_with"):
        names = ", ".join(plant["confusion_with"])
        lines.append(f"   易混淆：{names}")
    return "\n".join(lines)


def _format_dangerous_animal(animal: dict) -> str:
    morph = animal.get("morphology", {})
    lines = [f"{animal['common_names']['zh-TW']} ({animal['scientific_name']})"]
    lines.append(f"   毒性：{animal.get('venom_type', 'N/A')}")
    field_map = {
        "body_color": "體色", "tail": "尾部", "head_shape": "頭形",
        "pupil": "瞳孔", "pit_organ": "頰窩", "body_size": "體型",
        "scales": "鱗片", "hood": "頸部", "dorsal_scales": "背鱗",
    }
    for key, label in field_map.items():
        if key in morph:
            lines.append(f"   {label}：{morph[key]}")
    lines.append(f"   棲地：{animal.get('habitat', 'N/A')}")
    lines.append(f"   行為：{animal.get('behavior', 'N/A')}")
    return "\n".join(lines)


def build_danger_screening_prompt(
    context: EnvironmentContext, kb: KnowledgeBase, target_type: str = "plant",
) -> str:
    header = context.to_prompt_header()
    threshold = THRESHOLDS["danger_screening"]

    if target_type == "plant":
        role = "a toxicology expert specializing in field identification of dangerous plants"
        target_word = "plant"
        species_list = "\n\n".join(
            f"{i+1}. {_format_toxic_plant(p)}" for i, p in enumerate(kb.toxic_plants)
        )
    else:
        role = "a herpetology and zoology expert specializing in identification of dangerous animals"
        target_word = "animal/snake"
        species_list = "\n\n".join(
            f"{i+1}. {_format_dangerous_animal(a)}" for i, a in enumerate(kb.dangerous_animals)
        )

    prompt = f"""{header}

You are {role}.
Analyze the {target_word} in the provided photo(s).

Follow these steps:

Step 1: Carefully observe the photo(s). Describe the morphological features you can see.

Step 2: Compare your observations against each species in the list below.

Step 3: For each species, report similarity (0-100%), matching features, and non-matching features.

Step 4: If any feature cannot be determined from the photo, explicitly state so.

Below is a list of toxic/dangerous species commonly found in this region:

{species_list}

If ANY species scores ≥ {threshold}% similarity, issue a clear warning.

IMPORTANT — OUTPUT FORMAT:
After completing your analysis (Steps 1-4), output a final summary
as a valid JSON object wrapped between {JSON_START_MARKER} and {JSON_END_MARKER}.

{JSON_START_MARKER}
{{
  "reasoning_summary": "用一段繁體中文簡述你的分析過程和判斷依據",
  "identification": {{
    "primary_match": {{
      "common_name_zh": "中文名",
      "common_name_en": "English name",
      "scientific_name": "學名",
      "confidence": 85
    }},
    "other_candidates": [
      {{"common_name_zh": "其他候選", "confidence": 10}}
    ],
    "observed_features": ["觀察到的特徵1", "觀察到的特徵2"],
    "unobservable_features": ["無法從照片判斷的特徵"]
  }},
  "danger_assessment": {{
    "is_dangerous": true,
    "danger_level": "high",
    "toxicity_description": "毒性描述",
    "affected_parts": ["受影響部位"],
    "symptoms": ["症狀"]
  }},
  "safety_action": {{
    "warning_level": "RED",
    "action": "DO_NOT_TOUCH",
    "message_zh": "安全警告訊息（繁體中文）",
    "first_aid_zh": "急救建議"
  }},
  "confusion_pairs": [
    {{
      "safe_species": "可能混淆的安全物種名",
      "dangerous_species": "對應的危險物種名",
      "distinguishing_method": "區分方式"
    }}
  ],
  "requires_interactive_test": false,
  "interactive_tests": []
}}
{JSON_END_MARKER}

Rules for JSON values:
- "confidence": integer 0-100
- "danger_level": one of "none", "low", "medium", "high", "critical"
- "warning_level": one of "GREEN", "YELLOW", "RED"
- "action": one of "SAFE_TO_USE", "CAUTION", "DO_NOT_TOUCH", "DO_NOT_EAT", "EVACUATE"
- All Chinese text fields MUST use Traditional Chinese (繁體中文)

First write your full Step 1-4 analysis in Traditional Chinese,
THEN output the JSON between {JSON_START_MARKER} and {JSON_END_MARKER}.
Do NOT skip the analysis steps.

Please respond in Traditional Chinese (繁體中文)."""

    return prompt


def build_confusion_pairs_prompt(context: EnvironmentContext, pair: dict) -> str:
    header = context.to_prompt_header()
    threshold = THRESHOLDS["confusion_pairs"]
    safe = pair["safe_species"]
    danger = pair["dangerous_species"]
    safe_features = "\n".join(f"- {k}：{v}" for k, v in safe["key_features"].items())
    danger_features = "\n".join(f"- {k}：{v}" for k, v in danger["key_features"].items())
    comparison_rows = ""
    for row in pair.get("comparison_table", []):
        comparison_rows += f"| {row['feature']} | {row['safe']} | {row['danger']} |\n"

    prompt = f"""{header}

你是一位植物學家，專精於野外求生中的危險植物鑑別。
這張照片中的植物可能是以下兩種之一，它們外觀非常相似，
但一個可食用，一個有毒。誤判可能致命。

【可食用】{safe['common_name_zh']} ({safe['scientific_name']})
{safe_features}

【有毒】{danger['common_name_zh']} ({danger['scientific_name']})
{danger_features}

【關鍵鑑別對照表】
| 特徵 | {safe['common_name_zh']}（可食） | {danger['common_name_zh']}（有毒） |
|------|------|------|
{comparison_rows}

請根據照片中可辨識的特徵，逐項比對上表，判斷：
1. 你觀察到哪些特徵？逐一描述
2. 每個特徵更符合{safe['common_name_zh']}還是{danger['common_name_zh']}？
3. 有哪些特徵無法從照片中判斷？
4. 最終判斷：更可能是哪一種？信心度（0-100%）？

⚠️ 重要規則：
- 如果信心度 < {threshold}%，請標注「無法確定，視為有毒，禁止食用」
- 如果任何關鍵鑑別特徵無法從照片確認，也請標注「無法確定，視為有毒」
- 在求生情境中，誤判比不判更危險

After your analysis, output a JSON summary:

{JSON_START_MARKER}
{{
  "confusion_pair_id": "{pair['id']}",
  "judgment": "safe" or "dangerous" or "uncertain",
  "confidence": 0,
  "safe_species": "{safe['common_name_zh']}",
  "dangerous_species": "{danger['common_name_zh']}",
  "feature_comparison": [
    {{
      "feature": "特徵名",
      "observation": "照片中觀察到的情況",
      "matches": "safe" or "dangerous" or "uncertain"
    }}
  ],
  "unobservable_features": ["無法判斷的特徵"],
  "requires_interactive_test": true,
  "recommended_tests": ["建議的互動測試"],
  "final_message_zh": "給使用者的最終建議（繁體中文）"
}}
{JSON_END_MARKER}

請用繁體中文回覆。"""
    return prompt


def build_useful_resources_prompt(context: EnvironmentContext, kb: KnowledgeBase) -> str:
    header = context.to_prompt_header()
    threshold = THRESHOLDS["useful_resources"]

    edible_list_parts = []
    for i, plant in enumerate(kb.edible_plants):
        morph = plant.get("morphology", {})
        lines = [f"{i+1}. {plant['common_names']['zh-TW']} ({plant['scientific_name']})"]
        lines.append(f"   食用部位：{', '.join(plant.get('edible_parts', []))}")
        for key, label in [
            ("leaf_shape", "葉形"), ("leaf_arrangement", "葉排列"),
            ("leaf_surface", "葉面"), ("flower", "花"), ("stem", "莖"),
            ("midrib", "中肋"), ("growth_pattern", "生長方式"),
            ("young_shoot", "嫩芽"), ("shape", "外形"), ("fruit", "果實"),
            ("underground", "地下部"), ("smell", "氣味"),
        ]:
            if key in morph:
                lines.append(f"   {label}：{morph[key]}")
        lines.append(f"   生長環境：{plant.get('habitat', 'N/A')}")
        harvesting = plant.get("harvesting", {})
        if harvesting:
            lines.append(f"   採集方式：{harvesting.get('method', 'N/A')}")
        prep = plant.get("preparation", {})
        if prep:
            lines.append(f"   食用方式：{prep.get('method', 'N/A')}")
        if plant.get("caution"):
            lines.append(f"   ⚠️ 注意：{plant['caution']}")
        edible_list_parts.append("\n".join(lines))

    edible_list = "\n\n".join(edible_list_parts)

    danger_names = "\n".join(
        f"- {p['common_names']['zh-TW']} ({p['scientific_name']}) — {p['toxicity'][:30]}..."
        for p in kb.toxic_plants[:5]
    )

    prompt = f"""{header}

You are a field survival botanist helping a person identify useful resources in the wild.
Analyze the plant in the provided photo(s).

Follow these steps:

Step 1: Describe the plant's visible morphological features.

Step 2: FIRST check against the dangerous plants list below.
If any match ≥ {THRESHOLDS['danger_screening']}%, issue a warning and STOP.

Step 3: If no danger match, compare against the useful plants list.

Step 4: For each useful species, report similarity (0-100%).

Step 5: If identified as useful (confidence ≥ {threshold}%), provide practical
instructions for safe harvesting and preparation.
If confidence < {threshold}%, say "無法確定，請勿採食".

=== DANGEROUS PLANTS (check first) ===
{danger_names}

=== USEFUL PLANTS ===
{edible_list}

After your analysis, output a JSON summary:

{JSON_START_MARKER}
{{
  "reasoning_summary": "分析摘要（繁體中文）",
  "danger_check": {{
    "passed": true,
    "matched_danger_species": null
  }},
  "identification": {{
    "primary_match": {{
      "common_name_zh": "中文名",
      "scientific_name": "學名",
      "confidence": 85
    }},
    "observed_features": ["特徵"]
  }},
  "usability": {{
    "is_useful": true,
    "edible_parts": ["可食部位"],
    "harvesting_method": "採集方式",
    "preparation_method": "食用/使用方式",
    "caution": "注意事項"
  }},
  "safety_action": {{
    "warning_level": "GREEN",
    "message_zh": "安全訊息（繁體中文）"
  }}
}}
{JSON_END_MARKER}

Please respond in Traditional Chinese (繁體中文)."""
    return prompt


def build_interactive_test_guidance(pair: dict) -> str:
    safe = pair["safe_species"]
    danger = pair["dangerous_species"]
    tests = pair.get("interactive_tests", [])
    if not tests:
        return f"⚠️ 此植物可能是{safe['common_name_zh']}或{danger['common_name_zh']}，無法確定，請勿食用。"
    lines = [
        f"⚠️ 此植物可能是{safe['common_name_zh']}（可食）或{danger['common_name_zh']}（有毒），",
        "僅靠照片無法區分。請執行以下測試：", "",
    ]
    for test in sorted(tests, key=lambda t: t.get("priority", 99)):
        critical = "（最關鍵！）" if test.get("is_critical") else ""
        lines.append(f"🔬 {test['test_name']}{critical}：")
        lines.append(f"   {test['instruction_zh']}")
        lines.append(f"   → {test['safe_result']}")
        lines.append(f"   → {test['danger_result']}")
        lines.append("")
    lines.extend([
        "📷 完成測試後，請拍攝測試結果照片上傳，我將結合新資訊重新判斷。",
        f"⚠️ 在完成測試前，請勿食用此植物。",
    ])
    return "\n".join(lines)


def build_multi_photo_wrapper(base_prompt: str, num_photos: int) -> str:
    photo_instructions = (
        f"I am providing {num_photos} photos of the SAME subject taken from different angles.\n"
        "Analyze ALL photos together to improve identification accuracy.\n"
        "For Step 1, describe features observed in EACH photo separately,\n"
        "then note which features are consistently observed across photos (higher reliability).\n\n"
    )
    return base_prompt.replace(
        "Step 1: Carefully observe the photo(s).",
        photo_instructions + "Step 1: For EACH photo, describe the morphological features you observe.",
    )


print("✅ Prompt Builder 載入完成")

In [ ]:
# Cell 8: Response Parser (回覆解析器)

@dataclass
class IdentificationResult:
    raw_response: str
    analysis_text: str
    json_data: Optional[dict]
    parse_success: bool
    parse_error: Optional[str] = None

    @property
    def primary_match_name(self) -> str:
        if not self.json_data: return "未知"
        return self.json_data.get("identification", {}).get("primary_match", {}).get("common_name_zh", "未知")

    @property
    def confidence(self) -> int:
        if not self.json_data: return 0
        return self.json_data.get("identification", {}).get("primary_match", {}).get("confidence", 0)

    @property
    def danger_level(self) -> str:
        if not self.json_data: return "unknown"
        return self.json_data.get("danger_assessment", {}).get("danger_level", "unknown")

    @property
    def warning_level(self) -> str:
        if not self.json_data: return "GRAY"
        return self.json_data.get("safety_action", {}).get("warning_level", "GRAY")

    @property
    def is_dangerous(self) -> bool:
        if not self.json_data: return False
        return self.json_data.get("danger_assessment", {}).get("is_dangerous", False)

    @property
    def safety_message(self) -> str:
        if not self.json_data: return "無法解析辨識結果。"
        return self.json_data.get("safety_action", {}).get("message_zh", "無安全訊息。")

    @property
    def requires_interactive_test(self) -> bool:
        if not self.json_data: return False
        return self.json_data.get("requires_interactive_test", False)

    @property
    def has_confusion_pairs(self) -> bool:
        if not self.json_data: return False
        pairs = self.json_data.get("confusion_pairs", [])
        return bool(pairs and any(p.get("safe_species") and p["safe_species"] != "N/A" for p in pairs))


@dataclass
class ConfusionResult:
    raw_response: str
    json_data: Optional[dict]
    parse_success: bool

    @property
    def judgment(self) -> str:
        if not self.json_data: return "uncertain"
        return self.json_data.get("judgment", "uncertain")

    @property
    def confidence(self) -> int:
        if not self.json_data: return 0
        return self.json_data.get("confidence", 0)

    @property
    def is_safe(self) -> bool:
        return self.judgment == "safe" and self.confidence >= THRESHOLDS["confusion_pairs"]

    @property
    def final_message(self) -> str:
        if not self.json_data: return "無法解析鑑別結果，視為有毒，禁止食用。"
        return self.json_data.get("final_message_zh", "無法確定，視為有毒，禁止食用。")

    @property
    def recommended_tests(self) -> list:
        if not self.json_data: return []
        return self.json_data.get("recommended_tests", [])


def _extract_json_from_markers(text: str) -> Optional[str]:
    if JSON_START_MARKER in text and JSON_END_MARKER in text:
        json_str = text.split(JSON_START_MARKER, 1)[1].split(JSON_END_MARKER, 1)[0]
        return json_str.strip()
    return None


def _extract_json_fallback(text: str) -> Optional[str]:
    matches = re.findall(r'\{[\s\S]*\}', text)
    if matches:
        return matches[-1].strip()
    return None


def _try_parse_json(json_str: str) -> tuple:
    try:
        clean = json_str.strip()
        if clean.startswith("```"):
            clean = clean.split("\n", 1)[1]
            clean = clean.rsplit("```", 1)[0]
        data = json.loads(clean)
        return data, None
    except json.JSONDecodeError as e:
        return None, f"JSON 解析失敗：{e}"


def parse_danger_screening_response(response: str) -> IdentificationResult:
    if JSON_START_MARKER in response:
        analysis_text = response.split(JSON_START_MARKER)[0].strip()
    else:
        analysis_text = response
    json_str = _extract_json_from_markers(response)
    if json_str is None:
        json_str = _extract_json_fallback(response)
    if json_str is None:
        return IdentificationResult(
            raw_response=response, analysis_text=analysis_text,
            json_data=None, parse_success=False, parse_error="找不到 JSON 區塊",
        )
    data, error = _try_parse_json(json_str)
    return IdentificationResult(
        raw_response=response, analysis_text=analysis_text,
        json_data=data, parse_success=data is not None, parse_error=error,
    )


def parse_confusion_pairs_response(response: str) -> ConfusionResult:
    json_str = _extract_json_from_markers(response)
    if json_str is None:
        json_str = _extract_json_fallback(response)
    if json_str is None:
        return ConfusionResult(raw_response=response, json_data=None, parse_success=False)
    data, _ = _try_parse_json(json_str)
    return ConfusionResult(raw_response=response, json_data=data, parse_success=data is not None)


def parse_useful_resources_response(response: str) -> IdentificationResult:
    return parse_danger_screening_response(response)


print("✅ Response Parser 載入完成")

In [ ]:
# Cell 9: Model Loader (Kaggle GPU 版)

class GemmaModel:
    def __init__(self, model_id: str = MODEL_ID, device: str = "auto"):
        self.model_id = model_id
        self.processor = AutoProcessor.from_pretrained(model_id)
        device_map = device
        if device == "auto" and torch.cuda.device_count() > 1:
            device_map = {"": "cuda:0"}
        dtype = getattr(torch, DEFAULT_DTYPE, torch.bfloat16)
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id, torch_dtype=dtype, device_map=device_map,
        )
        self._device = self.model.device

    def generate(self, prompt: str, images: Optional[list] = None, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
        content = []
        if images:
            for img in images:
                content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": prompt})
        messages = [{"role": "user", "content": content}]
        inputs = self.processor.apply_chat_template(
            messages, tokenize=True, return_dict=True,
            return_tensors="pt", add_generation_prompt=True,
        ).to(self._device)
        input_len = inputs["input_ids"].shape[-1]
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        return self.processor.decode(outputs[0][input_len:], skip_special_tokens=True)


print("✅ Model Loader 定義完成（尚未載入模型）")

In [ ]:
# Cell 10: Pipeline (四層辨識核心)

@dataclass
class PipelineResult:
    layer_reached: int
    warning_level: str
    summary_zh: str
    details: dict = field(default_factory=dict)
    danger_result: Optional[IdentificationResult] = None
    confusion_result: Optional[ConfusionResult] = None
    useful_result: Optional[IdentificationResult] = None
    interactive_guidance: Optional[str] = None
    raw_responses: list = field(default_factory=list)


class JungleSurvivorPipeline:
    def __init__(self, model):
        self.model = model

    def identify(
        self, images: list, context: Optional[EnvironmentContext] = None, mode: str = "auto",
    ) -> PipelineResult:
        if context is None:
            context = create_context()
        kb = load_knowledge_base_embedded()
        target_type = "animal" if mode == "animal" else "plant"

        # Layer 1: 危險物種快篩
        danger_result = self._run_danger_screening(images, context, kb, target_type)
        if not danger_result.parse_success:
            return PipelineResult(
                layer_reached=1, warning_level="GRAY",
                summary_zh="⚠️ 辨識結果解析失敗，請重新拍照嘗試。",
                danger_result=danger_result, raw_responses=[danger_result.raw_response],
            )

        if danger_result.is_dangerous and danger_result.confidence >= THRESHOLDS["danger_screening"]:
            result = PipelineResult(
                layer_reached=1, warning_level="RED",
                summary_zh=(
                    f"🔴 警告：偵測到危險物種！\n"
                    f"辨識為：{danger_result.primary_match_name}\n"
                    f"信心度：{danger_result.confidence}%\n"
                    f"{danger_result.safety_message}"
                ),
                danger_result=danger_result, raw_responses=[danger_result.raw_response],
            )
            # Layer 2: 混淆物種鑑別
            if danger_result.has_confusion_pairs:
                confusion_result, guidance = self._run_confusion_check(images, context, kb, danger_result)
                if confusion_result:
                    result.confusion_result = confusion_result
                    result.raw_responses.append(confusion_result.raw_response)
                    result.layer_reached = 2
                    if confusion_result.is_safe:
                        result.warning_level = "GREEN"
                        result.summary_zh = f"🟢 經混淆鑑別確認為安全物種\n{confusion_result.final_message}"
                    else:
                        result.interactive_guidance = guidance
                        result.summary_zh += f"\n\n🔬 此物種與可食物種外觀相似，需要進一步確認：\n{guidance}"
            return result

        if mode == "danger_only" or mode == "animal":
            return PipelineResult(
                layer_reached=1, warning_level="GREEN",
                summary_zh=(
                    f"🟢 未匹配到已知危險物種。\n"
                    f"最高相似度物種：{danger_result.primary_match_name}"
                    f"（{danger_result.confidence}%，低於警告閾值 {THRESHOLDS['danger_screening']}%）"
                ),
                danger_result=danger_result, raw_responses=[danger_result.raw_response],
            )

        # Layer 3: 可利用資源辨識
        useful_result = self._run_useful_resources(images, context, kb)
        if useful_result.parse_success and useful_result.confidence >= THRESHOLDS["useful_resources"]:
            usability = useful_result.json_data.get("usability", {}) if useful_result.json_data else {}
            harvesting = usability.get("harvesting_method", "")
            preparation = usability.get("preparation_method", "")
            caution = usability.get("caution", "")
            usage_info = ""
            if harvesting: usage_info += f"\n📋 採集方式：{harvesting}"
            if preparation: usage_info += f"\n🍳 食用方式：{preparation}"
            if caution: usage_info += f"\n⚠️ 注意：{caution}"
            return PipelineResult(
                layer_reached=3, warning_level="GREEN",
                summary_zh=(
                    f"🟢 辨識為可利用資源\n"
                    f"物種：{useful_result.primary_match_name}\n"
                    f"信心度：{useful_result.confidence}%{usage_info}"
                ),
                useful_result=useful_result, danger_result=danger_result,
                raw_responses=[danger_result.raw_response, useful_result.raw_response],
            )

        # Layer 4: 無法確定
        return PipelineResult(
            layer_reached=4, warning_level="GRAY",
            summary_zh="⚪ 無法確定此物種的身份或用途。\n建議不要接觸或食用。\n如有可能，請拍攝更多角度的照片重試。",
            danger_result=danger_result,
            useful_result=useful_result if useful_result else None,
            raw_responses=[danger_result.raw_response, useful_result.raw_response if useful_result else ""],
        )

    def _run_danger_screening(self, images, context, kb, target_type) -> IdentificationResult:
        prompt = build_danger_screening_prompt(context, kb, target_type)
        if len(images) > 1:
            prompt = build_multi_photo_wrapper(prompt, len(images))
        response = self.model.generate(prompt, images)
        return parse_danger_screening_response(response)

    def _run_confusion_check(self, images, context, kb, danger_result):
        if not danger_result.json_data:
            return None, None
        matched_id = (
            danger_result.json_data.get("identification", {})
            .get("primary_match", {}).get("scientific_name", "").lower().replace(" ", "_")
        )
        matching_pair = None
        for pair in kb.confusion_pairs:
            safe_id = pair["safe_species"].get("id", "")
            danger_id = pair["dangerous_species"].get("id", "")
            if matched_id in (safe_id, danger_id):
                matching_pair = pair
                break
            safe_name = pair["dangerous_species"].get("scientific_name", "").lower().replace(" ", "_")
            if matched_id == safe_name:
                matching_pair = pair
                break
        if not matching_pair:
            return None, None
        prompt = build_confusion_pairs_prompt(context, matching_pair)
        response = self.model.generate(prompt, images)
        confusion_result = parse_confusion_pairs_response(response)
        guidance = None
        if not confusion_result.is_safe:
            guidance = build_interactive_test_guidance(matching_pair)
        return confusion_result, guidance

    def _run_useful_resources(self, images, context, kb) -> IdentificationResult:
        prompt = build_useful_resources_prompt(context, kb)
        if len(images) > 1:
            prompt = build_multi_photo_wrapper(prompt, len(images))
        response = self.model.generate(prompt, images)
        return parse_useful_resources_response(response)


print("✅ Pipeline 定義完成")

In [ ]:
# Cell 11: 載入模型 + 建立 Pipeline

print("正在載入 Gemma 4 E2B 模型...")
model = GemmaModel()
pipeline = JungleSurvivorPipeline(model)
print("✅ 模型載入完成！Pipeline 已就緒。")

In [ ]:
# Cell 12: 工具函式 — 圖片載入 + 結果顯示

def load_image_from_url(url: str) -> Image.Image:
    """從 URL 載入圖片"""
    response = requests.get(url, timeout=30, headers={
        "User-Agent": "Mozilla/5.0 (compatible; JungleSurvivor/1.0)"
    })
    response.raise_for_status()
    img = Image.open(BytesIO(response.content)).convert("RGB")
    print(f"✅ 圖片載入成功：{url[:60]}... ({img.size[0]}x{img.size[1]})")
    return img


def display_result(result: PipelineResult, test_name: str):
    """格式化顯示 Pipeline 結果"""
    level_label = WARNING_LEVELS.get(result.warning_level, {}).get("label_zh", "⚪ 不確定")
    print(f"\n{'='*60}")
    print(f"  {test_name}")
    print(f"{'='*60}")
    print(f"  警告等級：{level_label}")
    print(f"  到達層級：Layer {result.layer_reached}")
    print(f"{'─'*60}")
    print(result.summary_zh)
    print(f"{'─'*60}")

    if result.danger_result and result.danger_result.parse_success:
        dr = result.danger_result
        print(f"\n  📊 危險快篩結果：")
        print(f"     辨識物種：{dr.primary_match_name}")
        print(f"     信心度：{dr.confidence}%")
        print(f"     危險等級：{dr.danger_level}")
        print(f"     JSON 解析：{'✅ 成功' if dr.parse_success else '❌ 失敗'}")

    if result.confusion_result:
        cr = result.confusion_result
        print(f"\n  🔬 混淆鑑別結果：")
        print(f"     判定：{cr.judgment}")
        print(f"     信心度：{cr.confidence}%")
        print(f"     安全：{'✅ 是' if cr.is_safe else '❌ 否'}")

    if result.useful_result and result.useful_result.parse_success:
        ur = result.useful_result
        print(f"\n  🌿 可利用資源結果：")
        print(f"     辨識物種：{ur.primary_match_name}")
        print(f"     信心度：{ur.confidence}%")

    if result.interactive_guidance:
        print(f"\n  🔬 互動測試指引：")
        print(result.interactive_guidance)

    print(f"\n{'='*60}\n")


print("✅ 工具函式載入完成")

## 整合測試

以下 4 個測試涵蓋完整 Pipeline 的所有層級：

| 測試 | 對象 | 預期結果 | 驗證目標 |
|------|------|----------|----------|
| Test 1 | 姑婆芋（單張照片） | 🔴 危險警告 + 混淆鑑別觸發 | Layer 1→2、JSON 解析 |
| Test 2 | 姑婆芋（多張照片） | 信心度提升至 ≥90% | 多照片策略 |
| Test 3 | 赤尾青竹絲 | 🔴 毒蛇警告 | 動物辨識模式 |
| Test 4 | 山蘇 | 🟢 可食用 + 採集指引 | Layer 3、完整流程 |

In [ ]:
# Test 1: 危險植物快篩 — 姑婆芋（單張照片）
# 預期：🔴 危險警告，信心度 ≥ 60%，觸發混淆鑑別 + 互動測試

print("🧪 Test 1: 危險植物快篩 — 姑婆芋（單張照片）")
print("-" * 40)

img_alocasia = load_image_from_url("https://flora.naturestore.com.tw/wp-content/uploads/P0115.jpg")
ctx = create_context(country="台灣", altitude=500, vegetation_zone="低海拔闊葉林")

result_1 = pipeline.identify([img_alocasia], ctx, mode="auto")
display_result(result_1, "Test 1: 姑婆芋 單張快篩")

# 驗證
assert result_1.danger_result.parse_success, "❌ JSON 解析失敗"
assert result_1.warning_level == "RED", f"❌ 預期 RED，實際 {result_1.warning_level}"
assert result_1.danger_result.confidence >= THRESHOLDS["danger_screening"], \
    f"❌ 信心度 {result_1.danger_result.confidence}% 低於閾值"
print("✅ Test 1 通過！")

In [ ]:
# Test 2: 多照片辨識 — 姑婆芋（兩張不同角度）
# 預期：信心度 ≥ 90%（高於 Test 1 的單張）

print("🧪 Test 2: 多照片辨識 — 姑婆芋（兩張照片）")
print("-" * 40)

img_alocasia_1 = load_image_from_url("https://flora.naturestore.com.tw/wp-content/uploads/P0115.jpg")
img_alocasia_2 = load_image_from_url("https://i0.wp.com/ttcbg.com/wp-content/uploads/2024/08/alocasia-odora-00.jpg")
ctx = create_context(country="台灣", altitude=500, vegetation_zone="低海拔闊葉林")

result_2 = pipeline.identify([img_alocasia_1, img_alocasia_2], ctx, mode="auto")
display_result(result_2, "Test 2: 姑婆芋 多照片辨識")

# 驗證
assert result_2.danger_result.parse_success, "❌ JSON 解析失敗"
assert result_2.danger_result.confidence >= 85, \
    f"❌ 多照片信心度 {result_2.danger_result.confidence}% 未達預期 ≥85%"
print(f"✅ Test 2 通過！信心度 {result_2.danger_result.confidence}%"
      f"（單張 Test1: {result_1.danger_result.confidence}%）")

In [ ]:
# Test 3: 蛇類辨識 — 赤尾青竹絲
# 預期：使用 animal 模式，信心度 ≥ 60% 觸發警告

print("🧪 Test 3: 蛇類辨識 — 赤尾青竹絲")
print("-" * 40)

img_snake = load_image_from_url("https://g.udn.com.tw/upfiles/B_CO/connieherbest/PSN_PHOTO/634/f_10023634_1.JPG")
ctx = create_context(country="台灣", altitude=500, vegetation_zone="低海拔闊葉林")

result_3 = pipeline.identify([img_snake], ctx, mode="animal")
display_result(result_3, "Test 3: 蛇類辨識")

# 驗證
assert result_3.danger_result.parse_success, "❌ JSON 解析失敗"
print(f"✅ Test 3 完成！辨識為：{result_3.danger_result.primary_match_name}，"
      f"信心度 {result_3.danger_result.confidence}%")

In [ ]:
# Test 4: 可食用植物辨識 — 山蘇
# 預期：先通過危險快篩（無匹配）→ 進入 Layer 3 → 辨識為山蘇 ≥ 70%

print("🧪 Test 4: 可食用植物辨識 — 山蘇")
print("-" * 40)

img_fern = load_image_from_url("https://kmweb.moa.gov.tw/files/IMITA_Gallery/166/9a2fae37fe_m.jpg")
ctx = create_context(country="台灣", altitude=800, vegetation_zone="中海拔闊葉林")

result_4 = pipeline.identify([img_fern], ctx, mode="auto")
display_result(result_4, "Test 4: 山蘇 可食用辨識")

# 驗證
if result_4.useful_result and result_4.useful_result.parse_success:
    assert result_4.useful_result.confidence >= THRESHOLDS["useful_resources"], \
        f"❌ 信心度 {result_4.useful_result.confidence}% 低於閾值"
    print(f"✅ Test 4 通過！辨識為：{result_4.useful_result.primary_match_name}，"
          f"信心度 {result_4.useful_result.confidence}%")
else:
    print(f"ℹ️ Test 4 結果：Layer {result_4.layer_reached}，{result_4.warning_level}")

In [ ]:
# 測試總結

print("\n" + "=" * 60)
print("  📋 整合測試總結")
print("=" * 60)

tests = [
    ("Test 1: 姑婆芋（單張）", result_1),
    ("Test 2: 姑婆芋（多張）", result_2),
    ("Test 3: 蛇類辨識", result_3),
    ("Test 4: 山蘇（可食用）", result_4),
]

for name, r in tests:
    level = WARNING_LEVELS.get(r.warning_level, {}).get("label_zh", "⚪")
    dr = r.danger_result
    json_ok = "✅" if (dr and dr.parse_success) else "❌"
    conf = dr.confidence if dr else 0
    species = dr.primary_match_name if dr else "N/A"
    if r.useful_result and r.useful_result.parse_success:
        species = r.useful_result.primary_match_name
        conf = r.useful_result.confidence
    print(f"  {name:30s} | {level:8s} | L{r.layer_reached} | {species:10s} | {conf:3d}% | JSON:{json_ok}")

print("\n" + "=" * 60)
print("  🎉 所有測試完成！")
print("=" * 60)